# 06. Mô Hình Nâng Cao, Tinh Chỉnh Siêu Tham Số & Dự Đoán Sớm với 5-Fold Stratified Group CV
## Giai đoạn 7B (Tuần 7): 5 Mô hình Nâng cao, Hyperparameter Tuning, In-Fold $\tau^*_k$, Early Prediction (25–100%) & Phân loại Đa lớp 4-Class
**Dự án**: LMS Tiên tiến và Learning Analytics (Bộ dữ liệu OULAD)  
**Tác giả**: Vũ Hoà Bình (MSV: 202416137)  

---

### Kiến trúc & Quy chuẩn Thực nghiệm 5-Fold Stratified Group CV (Zero Data Leakage):
1. **5 Mô hình Nâng cao theo Đề cương**:
   - **XGBoost** (`XGBClassifier`), **LightGBM** (`LGBMClassifier`), **CatBoost / HistGradientBoosting** (tự động dùng `CatBoostClassifier` nếu hệ thống cho phép nạp DLL, hoặc `HistGradientBoostingClassifier` tương đương khi Windows Application Control chặn `_catboost.pyd`), **Neural Network** (`MLPClassifier` chuẩn hóa bổ sung trong Fold), và **Ensemble / Stacking** (`StackingClassifier` kết hợp LightGBM + XGBoost + HistGBM + Random Forest với meta-learner `LogisticRegression`).
2. **Đối sánh 10 Mô hình & Kiểm định Thống kê**:
   - Hợp nhất kết quả 5 Baseline từ `week7_baseline_cv_metrics.json` với 5 mô hình Nâng cao trên cùng phân vùng **5-Fold Stratified Group CV** (`id_student`), thực hiện kiểm định **Paired $t$-test** (trên 5 Fold) và **McNemar's Test** (trên $100\%$ dự đoán Out-of-Fold).
3. **Hyperparameter Tuning (`RandomizedSearchCV`) & Tối ưu Ngưỡng Cảnh báo Sớm In-Fold ($\tau^*_k$)**:
   - Tinh chỉnh siêu tham số bằng `RandomizedSearchCV` lồng trong Fold (`StratifiedGroupKFold(n_splits=3)`) tối ưu `ROC-AUC` / `Recall`.
   - Trong từng Fold $k \in \{1..5\}$, tách nội bộ $85\%$ Inner-Train / $15\%$ Inner-Val theo `id_student` để học ngưỡng quyết định $\tau^*_k$ sao cho $\text{Recall}^{\text{inner-val}}(\tau) \ge 0.76$ với $F_1$ tối đa, rồi chấm duy nhất 1 lần lên Test Fold $k$.
4. **Early Prediction (25%, 50%, 75%, 100%) & Mở rộng Đa lớp 4-Class**:
   - Thử nghiệm toàn diện qua 4 lát cắt thời lượng trên `dynamic_active` và `full`, đánh giá thêm bài toán phân loại 4 nhãn (`Distinction`, `Pass`, `Fail`, `Withdrawn`) và cầu nối xác suất gộp.
   - Đóng gói mô hình tốt nhất (`models/best_model.pkl`, `models/best_model_5fold_ensemble.joblib`, `models/best_model_metadata.json`), lưu `week7_experiment_metrics.json` và `week7_cv_results_summary.csv` vào `data/processed/week7_modeling/`, xuất Hình 34–36 và chạy **10/10 Unit Tests**.

In [1]:
import json
import logging
import sys
import time
import unittest
import warnings
from pathlib import Path
from typing import Any, Dict, List, Tuple

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from lightgbm import LGBMClassifier
from scipy import stats
from sklearn.base import clone
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import RandomizedSearchCV, StratifiedGroupKFold, StratifiedKFold, cross_val_predict
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler, StandardScaler
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

if sys.platform == "win32":
    try:
        sys.stdout.reconfigure(encoding="utf-8")
        sys.stderr.reconfigure(encoding="utf-8")
    except Exception:
        pass

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)],
    force=True,
)
logger = logging.getLogger("OULAD_Week7_Advanced")

plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
plt.rcParams["font.family"] = "DejaVu Sans"
plt.rcParams["font.sans-serif"] = "DejaVu Sans"
plt.rcParams["axes.unicode_minus"] = False
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: "%.4f" % x)

# Kiểm tra khả năng nạp CatBoost trên hệ điều hành hiện tại (xử lý chính sách Windows AppControl)
CATBOOST_AVAILABLE = False
CATBOOST_STATUS_MSG = ""
try:
    from catboost import CatBoostClassifier  # type: ignore
    CATBOOST_AVAILABLE = True
    CATBOOST_STATUS_MSG = "Native CatBoostClassifier loaded successfully."
except (ImportError, OSError) as exc:
    CATBOOST_AVAILABLE = False
    CATBOOST_STATUS_MSG = (
        f"CatBoost DLL bị chặn hoặc không khả dụng ({type(exc).__name__}: {exc}); "
        "tự động kích hoạt HistGradientBoostingClassifier tương đương (L2 histogram boosting)."
    )

PROJECT_ROOT = Path("..").resolve() if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
SNAPSHOTS_DIR = PROCESSED_DIR / "week5_6_snapshots"
WEEK7_PROCESSED_DIR = PROCESSED_DIR / "week7_modeling"
REPORTS_DIR = PROJECT_ROOT / "reports"
FIGURES_DIR = REPORTS_DIR / "figures"
MODELS_DIR = PROJECT_ROOT / "models"

for d in [PROCESSED_DIR, SNAPSHOTS_DIR, WEEK7_PROCESSED_DIR, REPORTS_DIR, FIGURES_DIR, MODELS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Project Root : {PROJECT_ROOT}")
print(f"Processed Dir: {PROCESSED_DIR}")
print(f"Models Dir   : {MODELS_DIR}")
print(f"CatBoost Stat: {CATBOOST_STATUS_MSG}")

Project Root : C:\Users\mrore\Project_I
Processed Dir: C:\Users\mrore\Project_I\data\processed
Models Dir   : C:\Users\mrore\Project_I\models
CatBoost Stat: Native CatBoostClassifier loaded successfully.


## 1. Động cơ `OULAD5FoldCVEngine` & Bộ Khởi Tạo 5 Mô Hình Nâng Cao
Bảo đảm tính tự chủ $100\%$ cho notebook: tích hợp lại lớp chia 5 phần `StratifiedGroupKFold(n_splits=5)` theo `id_student`, hàm tiền xử lý In-Fold `prepare_fold_matrices`, hàm tối ưu ngưỡng In-Fold $\tau^*_k$, hàm kiểm định thống kê (`paired_t_test` và `mcnemar_test`) và danh sách 5 mô hình nâng cao.

In [2]:
class OULAD5FoldCVEngine:
    """
    Động cơ 5-Fold Stratified Group CV (chia 5 phần đều nhau theo id_student)
    và tiền xử lý In-Fold Zero Data Leakage cho bộ 114 biến week6_advanced.
    """

    ALL_ACTIVITY_TYPES = [
        "dataplus", "dualpane", "externalquiz", "folder", "forumng",
        "glossary", "homepage", "htmlactivity", "oucollaborate", "oucontent",
        "ouelluminate", "ouwiki", "page", "questionnaire", "quiz",
        "repeatactivity", "resource", "sharedsubpage", "subpage", "url",
    ]

    ORDINAL_MAPPINGS = {
        "highest_education": {
            "No Formal quals": 0,
            "Lower Than A Level": 1,
            "A Level or Equivalent": 2,
            "HE Qualification": 3,
            "Post Graduate Qualification": 4,
        },
        "imd_band": {
            "0-10%": 0, "10-20": 1, "20-30%": 2, "30-40%": 3, "40-50%": 4,
            "50-60%": 5, "60-70%": 6, "70-80%": 7, "80-90%": 8, "90-100%": 9,
            "Unknown": 4,
        },
        "age_band": {"0-35": 0, "35-55": 1, "55<=": 2},
    }

    CLASS_4_MAP = {"Distinction": 0, "Pass": 1, "Fail": 2, "Withdrawn": 3}
    NOMINAL_COLS = ["gender", "disability", "region", "code_module"]
    ORDINAL_COLS = ["highest_education", "imd_band", "age_band"]
    FLAG_COLS = ["imd_was_missing", "has_no_submissions", "has_no_clicks"]

    @classmethod
    def get_week6_numeric_cols(cls) -> List[str]:
        all_20_act_cols = [f"clicks_{act}" for act in cls.ALL_ACTIVITY_TYPES]
        return [
            "total_clicks", "active_days", "clicks_per_day", "clicks_per_active_day",
            "click_std", "click_cv", "active_days_in_course", "active_days_ratio",
            "pre_course_clicks", "has_pre_course_clicks", "active_span_days",
            *all_20_act_cols,
            "cognitive_clicks", "social_clicks", "oucontent_ratio", "forumng_ratio",
            "quiz_ratio", "resource_ratio", "glossary_ratio", "cognitive_ratio",
            "social_ratio", "activity_diversity", "activity_entropy",
            "total_course_assessments", "due_course_assessments", "avg_score",
            "weighted_avg_score", "earned_weight_score", "min_score", "max_score",
            "score_std", "first_score", "num_submissions", "num_submissions_raw",
            "submission_rate", "course_progress_submission_rate", "avg_days_before_deadline",
            "late_submissions", "late_submission_rate", "tma1_score", "tma1_days_early",
            "submitted_tma1",
            "num_of_prev_attempts", "studied_credits", "module_presentation_length",
            "days_before_start", "is_repeat_student", "credit_overload", "is_late_registration",
            "clicks_week_1_to_4", "clicks_last_2_weeks", "clicks_trend", "engagement_decay",
            "days_since_last_activity", "inactivity_ratio", "max_inactivity_streak", "consistency_score",
            "active_weeks", "login_regularity", "weekend_clicks", "weekend_ratio",
            "burst_count", "burst_ratio",
            "last_minus_first_score", "score_trend_slope",
        ]

    @staticmethod
    def filter_cohort(df: pd.DataFrame, cohort_type: str = "dynamic_active") -> pd.DataFrame:
        if cohort_type == "full":
            return df.copy().reset_index(drop=True)
        elif cohort_type == "pre_course_active":
            mask = df["date_unregistration"].isna() | (df["date_unregistration"] >= 0)
            return df[mask].copy().reset_index(drop=True)
        elif cohort_type == "dynamic_active":
            mask = df["date_unregistration"].isna() | (df["date_unregistration"] > df["cutoff_day"])
            return df[mask].copy().reset_index(drop=True)
        raise ValueError(f"cohort_type không hợp lệ: {cohort_type}")

    @staticmethod
    def get_5fold_splits(
        df: pd.DataFrame, n_splits: int = 5, random_state: int = 42
    ) -> List[Tuple[int, np.ndarray, np.ndarray]]:
        sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
        y = df["at_risk"].values
        groups = df["id_student"].values
        return [(fold, tr_idx, te_idx) for fold, (tr_idx, te_idx) in enumerate(sgkf.split(df, y, groups=groups), start=1)]

    def fit_preprocessor(self, train_df: pd.DataFrame) -> Dict[str, Any]:
        """Chỉ fit Imputation, Outlier Bounds, OneHotEncoder và RobustScaler trên tập Train."""
        num_cols = self.get_week6_numeric_cols()
        imd_mode = (
            train_df["imd_band"].mode().iloc[0]
            if not train_df["imd_band"].mode().empty
            else "20-30%"
        )
        reg_median = float(train_df["date_registration"].median())
        skewed_cols = [
            c for c in num_cols
            if (
                (c.startswith("clicks_") and c != "clicks_trend")
                or c in [
                    "total_clicks", "active_days", "active_days_in_course",
                    "daily_click_mean", "daily_click_std", "click_std",
                    "pre_course_clicks", "cognitive_clicks", "social_clicks",
                    "weekend_clicks", "studied_credits",
                ]
            )
        ]
        bounds: Dict[str, Tuple[float, float]] = {}
        for col in skewed_cols:
            s = train_df[col].dropna()
            q25, q75 = float(s.quantile(0.25)), float(s.quantile(0.75))
            iqr = q75 - q25
            if iqr > 0:
                bounds[col] = (0.0, q75 + 3.0 * iqr)

        ohe = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
        ohe.fit(train_df[self.NOMINAL_COLS].fillna("Missing").astype(str))
        ohe_names = list(ohe.get_feature_names_out(self.NOMINAL_COLS))

        temp_state = {
            "num_cols": num_cols,
            "imd_mode": imd_mode,
            "reg_median": reg_median,
            "skewed_cols": skewed_cols,
            "bounds": bounds,
            "ohe": ohe,
            "ohe_names": ohe_names,
            "cols_to_scale": num_cols + self.ORDINAL_COLS,
            "scaler": None,
        }
        tr_clean = self._clean_df(train_df, temp_state)
        scaler = RobustScaler()
        scaler.fit(tr_clean[temp_state["cols_to_scale"]])
        temp_state["scaler"] = scaler
        return temp_state

    def _clean_df(self, df_in: pd.DataFrame, state: Dict[str, Any]) -> pd.DataFrame:
        d = df_in.copy()
        d["imd_was_missing"] = d["imd_band"].isna().astype(float)
        d["has_no_submissions"] = (d["num_submissions"].fillna(0) == 0).astype(float)
        d["has_no_clicks"] = (d["total_clicks"].fillna(0) == 0).astype(float)

        d["imd_band"] = d["imd_band"].fillna(state["imd_mode"])
        d["date_registration"] = d["date_registration"].fillna(state["reg_median"])
        d["days_before_start"] = -d["date_registration"]

        score_cols = [
            "avg_score", "weighted_avg_score", "earned_weight_score",
            "min_score", "max_score", "score_std", "first_score", "tma1_score",
        ]
        for sc in score_cols:
            if sc in d.columns:
                d[sc] = d[sc].fillna(0.0)

        for ec in ["avg_days_early", "avg_days_before_deadline", "tma1_days_early"]:
            if ec in d.columns:
                d[ec] = d[ec].fillna(-10.0)

        for c in state["num_cols"]:
            if c in d.columns:
                d[c] = d[c].fillna(0.0)

        for col, mapping in self.ORDINAL_MAPPINGS.items():
            d[col] = d[col].map(mapping).fillna(mapping.get("Unknown", 2)).astype(float)

        for col, (low, high) in state["bounds"].items():
            if col in d.columns:
                d[col] = d[col].clip(lower=low, upper=high)

        for col in state["skewed_cols"]:
            if col in d.columns and "click" in col and "ratio" not in col and "per" not in col and "cv" not in col:
                d[col] = np.log1p(np.maximum(0.0, d[col]))

        return d

    def transform_with_state(self, df_in: pd.DataFrame, state: Dict[str, Any]) -> pd.DataFrame:
        d_clean = self._clean_df(df_in, state)
        cols_to_scale = state["cols_to_scale"]
        scaled_arr = state["scaler"].transform(d_clean[cols_to_scale])
        scaled_df = pd.DataFrame(scaled_arr, columns=cols_to_scale, index=df_in.index)
        ohe_arr = state["ohe"].transform(df_in[self.NOMINAL_COLS].fillna("Missing").astype(str))
        ohe_df = pd.DataFrame(ohe_arr, columns=state["ohe_names"], index=df_in.index)
        flags_df = d_clean[self.FLAG_COLS].copy()
        out = pd.concat([scaled_df, ohe_df, flags_df], axis=1)
        return out.replace([np.inf, -np.inf], 0.0).fillna(0.0)

    def prepare_fold_matrices(
        self, train_df: pd.DataFrame, test_df: pd.DataFrame
    ) -> Tuple[pd.DataFrame, pd.DataFrame, Dict[str, Any]]:
        state = self.fit_preprocessor(train_df)
        return (
            self.transform_with_state(train_df, state),
            self.transform_with_state(test_df, state),
            state,
        )

    @staticmethod
    def evaluate_predictions(
        y_true: np.ndarray, y_prob: np.ndarray, threshold: float = 0.50
    ) -> Dict[str, Any]:
        y_pred = (y_prob >= threshold).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
        return {
            "accuracy": float(accuracy_score(y_true, y_pred)),
            "precision": float(precision_score(y_true, y_pred, zero_division=0)),
            "recall": float(recall_score(y_true, y_pred, zero_division=0)),
            "f1": float(f1_score(y_true, y_pred, zero_division=0)),
            "roc_auc": float(roc_auc_score(y_true, y_prob)),
            "pr_auc": float(average_precision_score(y_true, y_prob)),
            "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
            "brier": float(brier_score_loss(y_true, y_prob)),
            "tn": int(tn),
            "fp": int(fp),
            "fn": int(fn),
            "tp": int(tp),
        }


def tune_threshold_inner_val(
    y_val: np.ndarray, y_prob: np.ndarray, min_recall_target: float = 0.76
) -> Tuple[float, float, float]:
    """Tìm ngưỡng tau* tối ưu F1 trên Inner-Val sao cho Recall >= min_recall_target."""
    best_tau, best_f1, best_rec = 0.50, -1.0, 0.0
    for tau in np.arange(0.15, 0.71, 0.01):
        pred = (y_prob >= tau).astype(int)
        rec = float(recall_score(y_val, pred, zero_division=0))
        f1 = float(f1_score(y_val, pred, zero_division=0))
        if rec >= min_recall_target and f1 > best_f1:
            best_tau = round(float(tau), 2)
            best_f1 = f1
            best_rec = rec
    if best_f1 < 0:
        # Fallback nếu chưa đạt min_recall_target: chọn ngưỡng có Recall cao nhất rồi F1 cao nhất
        for tau in np.arange(0.15, 0.71, 0.01):
            pred = (y_prob >= tau).astype(int)
            rec = float(recall_score(y_val, pred, zero_division=0))
            f1 = float(f1_score(y_val, pred, zero_division=0))
            if (rec > best_rec) or (abs(rec - best_rec) < 1e-6 and f1 > best_f1):
                best_tau = round(float(tau), 2)
                best_f1 = f1
                best_rec = rec
    return best_tau, round(best_rec, 4), round(best_f1, 4)


def compute_mcnemar_test(
    y_true: np.ndarray, pred_a: np.ndarray, pred_b: np.ndarray
) -> Dict[str, Any]:
    """Kiểm định McNemar với hiệu chỉnh liên tục cho hai mô hình trên cùng tập OOF."""
    correct_a = pred_a == y_true
    correct_b = pred_b == y_true
    n01 = int(np.sum((~correct_a) & correct_b))  # A sai, B đúng
    n10 = int(np.sum(correct_a & (~correct_b)))  # A đúng, B sai
    denom = n01 + n10
    if denom == 0:
        return {"n01_A_wrong_B_right": n01, "n10_A_right_B_wrong": n10, "chi2": 0.0, "p_value": 1.0}
    chi2 = float(((abs(n01 - n10) - 1.0) ** 2) / denom)
    p_val = float(stats.chi2.sf(chi2, df=1))
    return {"n01_A_wrong_B_right": n01, "n10_A_right_B_wrong": n10, "chi2": round(chi2, 4), "p_value": p_val}


def build_advanced_models(scale_pos_weight: float = 1.4) -> Dict[str, Any]:
    """Khởi tạo 5 mô hình Nâng cao (Giai đoạn 7B)."""
    if CATBOOST_AVAILABLE:
        model_3 = CatBoostClassifier(
            iterations=200, depth=6, learning_rate=0.06, rsm=0.8, auto_class_weights="Balanced",
            random_seed=42, verbose=0, thread_count=-1,
        )
    else:
        model_3 = HistGradientBoostingClassifier(
            max_iter=220, max_depth=6, learning_rate=0.05, l2_regularization=1.0,
            class_weight="balanced", random_state=42,
        )

    lgbm_est = LGBMClassifier(
        n_estimators=220, num_leaves=31, learning_rate=0.05, subsample=0.8,
        colsample_bytree=0.8, class_weight="balanced", random_state=42, n_jobs=-1, verbose=-1,
    )
    xgb_est = XGBClassifier(
        n_estimators=220, max_depth=6, learning_rate=0.05, subsample=0.8,
        colsample_bytree=0.8, scale_pos_weight=scale_pos_weight, eval_metric="logloss",
        random_state=42, n_jobs=-1,
    )
    mlp_est = make_pipeline(
        StandardScaler(),
        MLPClassifier(
            hidden_layer_sizes=(128, 64), activation="relu", alpha=1e-3,
            batch_size=256, max_iter=60, early_stopping=True, validation_fraction=0.1,
            random_state=42,
        ),
    )

    stack_est = StackingClassifier(
        estimators=[
            ("lgbm", clone(lgbm_est)),
            ("xgb", clone(xgb_est)),
            ("hgb", clone(model_3)),
            ("rf", RandomForestClassifier(
                class_weight="balanced", n_estimators=150, max_depth=12, min_samples_leaf=5, n_jobs=-1, random_state=42
            )),
        ],
        final_estimator=LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42),
        cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=42),
        n_jobs=1,
    )

    return {
        "XGBoost": xgb_est,
        "LightGBM": lgbm_est,
        "CatBoost / HistGBM": model_3,
        "Neural Network (MLP)": mlp_est,
        "Ensemble / Stacking": stack_est,
    }


engine = OULAD5FoldCVEngine()
print("Đã khởi tạo OULAD5FoldCVEngine và 5 mô hình Nâng cao:", list(build_advanced_models().keys()))

Đã khởi tạo OULAD5FoldCVEngine và 5 mô hình Nâng cao: ['XGBoost', 'LightGBM', 'CatBoost / HistGBM', 'Neural Network (MLP)', 'Ensemble / Stacking']


## 2. Đánh Giá 5 Mô Hình Nâng Cao Qua 5-Fold Stratified Group CV & Đối Chiếu 10 Mô Hình Qua 4 Mốc (25%–100%)
Chạy đầy đủ 5 Fold (Fold 1 $\rightarrow$ Fold 5) cho cả 5 mô hình nâng cao trên cả 4 lát cắt `(25%, 50%, 75%, 100%, dynamic_active)` và `(25%, full)`, đồng thời ghép bảng so sánh **10 mô hình (5 Baseline từ Giai đoạn 7A + 5 Advanced)** qua cả 4 mốc và thực hiện **Paired $t$-test** & **McNemar's Test**.

In [3]:
snapshot_pcts = [25, 50, 75, 100]
snapshots: Dict[str, pd.DataFrame] = {}
for pct in snapshot_pcts:
    df_s = pd.read_csv(SNAPSHOTS_DIR / f"features_snapshot_{pct}pct.csv")
    snapshots[f"{pct}%"] = df_s
    logger.info(f"Đã nạp snapshot {pct}%: {df_s.shape[0]:,} x {df_s.shape[1]}")

baseline_path = WEEK7_PROCESSED_DIR / "week7_baseline_cv_metrics.json"
with open(baseline_path, "r", encoding="utf-8") as f:
    baseline_payload = json.load(f)
logger.info(f"Đã nạp kết quả Baseline 5-Fold CV từ {baseline_path.name}")

baseline_per_fold_df = pd.DataFrame(baseline_payload["per_fold_results"])
baseline_summary_df = pd.DataFrame(baseline_payload["summary_results"])
oof_base_25 = baseline_payload["oof_predictions"]["25%_dynamic_active"]
oof_base_100 = baseline_payload["oof_predictions"]["100%_dynamic_active"]

metric_keys = ["accuracy", "precision", "recall", "f1", "roc_auc", "pr_auc", "balanced_accuracy", "brier"]
adv_configs = [
    ("25%", "dynamic_active"),
    ("50%", "dynamic_active"),
    ("75%", "dynamic_active"),
    ("100%", "dynamic_active"),
    ("25%", "full"),
]

adv_per_fold_records: List[Dict[str, Any]] = []
adv_summary_records: List[Dict[str, Any]] = []
adv_oof_25_probs: Dict[str, np.ndarray] = {}
adv_oof_100_probs: Dict[str, np.ndarray] = {}

for snap_label, cohort_name in adv_configs:
    t0_cfg = time.time()
    df_c = engine.filter_cohort(snapshots[snap_label], cohort_name)
    y_all = df_c["at_risk"].values
    splits = engine.get_5fold_splits(df_c, n_splits=5, random_state=42)

    model_names = list(build_advanced_models().keys())
    oof_probs: Dict[str, np.ndarray] = {m: np.zeros(len(df_c), dtype=float) for m in model_names}
    model_fold_metrics: Dict[str, List[Dict[str, Any]]] = {m: [] for m in model_names}

    for fold, tr_idx, te_idx in splits:
        X_tr, X_te, _ = engine.prepare_fold_matrices(df_c.iloc[tr_idx], df_c.iloc[te_idx])
        y_tr, y_te = y_all[tr_idx], y_all[te_idx]
        pos_w = float((y_tr == 0).sum() / max(1, (y_tr == 1).sum()))

        builders = build_advanced_models(scale_pos_weight=pos_w)
        for m_name, clf in builders.items():
            t_s = time.time()
            clf.fit(X_tr, y_tr)
            prob_te = clf.predict_proba(X_te)[:, 1]
            fit_sec = time.time() - t_s

            oof_probs[m_name][te_idx] = prob_te
            m_res = engine.evaluate_predictions(y_te, prob_te, threshold=0.50)
            row = {
                "snapshot": snap_label,
                "cohort": cohort_name,
                "n_cohort": int(len(df_c)),
                "model": m_name,
                "fold": int(fold),
                "n_train": int(len(tr_idx)),
                "n_test": int(len(te_idx)),
                "fit_seconds": round(fit_sec, 3),
                **{k: round(m_res[k], 4) for k in metric_keys},
                "tn": m_res["tn"], "fp": m_res["fp"], "fn": m_res["fn"], "tp": m_res["tp"],
            }
            adv_per_fold_records.append(row)
            model_fold_metrics[m_name].append(row)

    for m_name in model_names:
        f_df = pd.DataFrame(model_fold_metrics[m_name])
        oof_m = engine.evaluate_predictions(y_all, oof_probs[m_name], threshold=0.50)
        s_row: Dict[str, Any] = {
            "snapshot": snap_label,
            "cohort": cohort_name,
            "n_cohort": int(len(df_c)),
            "at_risk_rate": round(float(y_all.mean()), 4),
            "model": m_name,
            "mean_fit_seconds": round(float(f_df["fit_seconds"].mean()), 3),
        }
        for k in metric_keys:
            s_row[f"{k}_mean"] = round(float(f_df[k].mean()), 4)
            s_row[f"{k}_sd"] = round(float(f_df[k].std(ddof=1)), 4)
            s_row[f"{k}_oof"] = round(float(oof_m[k]), 4)
        s_row["oof_tn"] = oof_m["tn"]
        s_row["oof_fp"] = oof_m["fp"]
        s_row["oof_fn"] = oof_m["fn"]
        s_row["oof_tp"] = oof_m["tp"]
        adv_summary_records.append(s_row)

    if snap_label == "25%" and cohort_name == "dynamic_active":
        adv_oof_25_probs = {m: oof_probs[m].copy() for m in model_names}
    elif snap_label == "100%" and cohort_name == "dynamic_active":
        adv_oof_100_probs = {m: oof_probs[m].copy() for m in model_names}

    logger.info(f"Hoàn tất 5-Fold CV 5 Mô hình Nâng cao cho ({snap_label}, {cohort_name}) trong {time.time() - t0_cfg:.1f}s")

adv_per_fold_df = pd.DataFrame(adv_per_fold_records)
adv_summary_df = pd.DataFrame(adv_summary_records)

# Hợp nhất bảng 10 mô hình (5 Baseline + 5 Advanced) tại 25% dynamic_active (tương thích ngược)
base_25_sum = baseline_summary_df[(baseline_summary_df["snapshot"] == "25%") & (baseline_summary_df["cohort"] == "dynamic_active")].copy()
base_25_sum["family"] = "Baseline (7A)"
adv_25_sum = adv_summary_df[(adv_summary_df["snapshot"] == "25%") & (adv_summary_df["cohort"] == "dynamic_active")].copy()
adv_25_sum["family"] = "Advanced (7B)"

combined_10_df = pd.concat([base_25_sum, adv_25_sum], ignore_index=True).sort_values("roc_auc_mean", ascending=False)
print("\n=== BẢNG ĐỐI SÁNH TOÀN DIỆN 10 MÔ HÌNH (5 BASELINE + 5 ADVANCED) — 5-FOLD CV TẠI MỐC 25% DYNAMIC ACTIVE ===")
display(combined_10_df[["family", "model", "accuracy_mean", "precision_mean", "recall_mean", "recall_sd", "f1_mean", "f1_sd", "roc_auc_mean", "roc_auc_sd", "pr_auc_mean", "brier_mean"]])

# Hợp nhất bảng 10 mô hình qua ĐỦ 4 MỐC (25%, 50%, 75%, 100%) trên dynamic_active
base_4snaps_sum = baseline_summary_df[
    (baseline_summary_df["cohort"] == "dynamic_active") & (baseline_summary_df["snapshot"].isin(["25%", "50%", "75%", "100%"]))
].copy()
base_4snaps_sum["family"] = "Baseline (7A)"
adv_4snaps_sum = adv_summary_df[
    (adv_summary_df["cohort"] == "dynamic_active") & (adv_summary_df["snapshot"].isin(["25%", "50%", "75%", "100%"]))
].copy()
adv_4snaps_sum["family"] = "Advanced (7B)"

combined_10_4snaps_df = pd.concat([base_4snaps_sum, adv_4snaps_sum], ignore_index=True)
combined_10_4snaps_df["_pct_order"] = combined_10_4snaps_df["snapshot"].str.replace("%", "", regex=False).astype(int)
combined_10_4snaps_df = combined_10_4snaps_df.sort_values(["_pct_order", "roc_auc_mean"], ascending=[True, False]).drop(columns=["_pct_order"]).reset_index(drop=True)
print("\n=== BẢNG ĐỐI SÁNH 10 MÔ HÌNH QUA CẢ 4 MỐC (25% -> 100% DYNAMIC ACTIVE, tau=0.50) ===")
display(combined_10_4snaps_df[["snapshot", "family", "model", "recall_mean", "f1_mean", "roc_auc_mean", "roc_auc_sd", "pr_auc_mean"]])

# Kiểm định thống kê: Paired t-test (trên 5 Fold) & McNemar's test (trên OOF 100% sinh viên tại 25% và 100%)
df_25_dyn = engine.filter_cohort(snapshots["25%"], "dynamic_active")
y_25_dyn = df_25_dyn["at_risk"].values
df_100_dyn = engine.filter_cohort(snapshots["100%"], "dynamic_active")
y_100_dyn = df_100_dyn["at_risk"].values

oof_y_map = {"25%": y_25_dyn, "100%": y_100_dyn}
oof_base_map = {"25%": oof_base_25, "100%": oof_base_100}
oof_adv_map = {"25%": adv_oof_25_probs, "100%": adv_oof_100_probs}

stat_tests = []
stat_tests_4snaps = []
for snap_lbl in ["25%", "50%", "75%", "100%"]:
    rf_folds_s = baseline_per_fold_df[
        (baseline_per_fold_df["snapshot"] == snap_lbl)
        & (baseline_per_fold_df["cohort"] == "dynamic_active")
        & (baseline_per_fold_df["model"] == "Random Forest")
    ].sort_values("fold")
    lr_folds_s = baseline_per_fold_df[
        (baseline_per_fold_df["snapshot"] == snap_lbl)
        & (baseline_per_fold_df["cohort"] == "dynamic_active")
        & (baseline_per_fold_df["model"] == "Logistic Regression")
    ].sort_values("fold")

    has_oof = snap_lbl in oof_base_map
    if has_oof:
        rf_oof_pred_s = (np.array(oof_base_map[snap_lbl]["probs"]["Random Forest"]) >= 0.50).astype(int)
        lr_oof_pred_s = (np.array(oof_base_map[snap_lbl]["probs"]["Logistic Regression"]) >= 0.50).astype(int)
        base_oof_preds = {"Random Forest": rf_oof_pred_s, "Logistic Regression": lr_oof_pred_s}
    else:
        base_oof_preds = {"Random Forest": None, "Logistic Regression": None}

    for adv_m in ["LightGBM", "XGBoost", "CatBoost / HistGBM", "Ensemble / Stacking"]:
        adv_f = adv_per_fold_df[
            (adv_per_fold_df["snapshot"] == snap_lbl)
            & (adv_per_fold_df["cohort"] == "dynamic_active")
            & (adv_per_fold_df["model"] == adv_m)
        ].sort_values("fold")
        adv_oof_pred_s = (oof_adv_map[snap_lbl][adv_m] >= 0.50).astype(int) if has_oof else None

        for base_label, base_f in [("Random Forest", rf_folds_s), ("Logistic Regression", lr_folds_s)]:
            t_auc, p_auc = stats.ttest_rel(adv_f["roc_auc"].values, base_f["roc_auc"].values)
            t_f1, p_f1 = stats.ttest_rel(adv_f["f1"].values, base_f["f1"].values)
            if has_oof:
                mc = compute_mcnemar_test(oof_y_map[snap_lbl], base_oof_preds[base_label], adv_oof_pred_s)
                mc_chi2, mc_p = mc["chi2"], mc["p_value"]
            else:
                mc_chi2, mc_p = None, None

            rec_stat = {
                "snapshot": snap_lbl,
                "advanced_model": adv_m,
                "baseline_model": base_label,
                "delta_auc_mean": round(float(adv_f["roc_auc"].mean() - base_f["roc_auc"].mean()), 4),
                "paired_t_auc_p": float(p_auc),
                "delta_f1_mean": round(float(adv_f["f1"].mean() - base_f["f1"].mean()), 4),
                "paired_t_f1_p": float(p_f1),
                "mcnemar_chi2": mc_chi2,
                "mcnemar_p": mc_p,
            }
            stat_tests_4snaps.append(rec_stat)
            if snap_lbl == "25%":
                stat_tests.append({k: v for k, v in rec_stat.items() if k != "snapshot"})

stat_tests_df = pd.DataFrame(stat_tests)
stat_tests_4snaps_df = pd.DataFrame(stat_tests_4snaps)
print("\n=== KIỂM ĐỊNH THỐNG KÊ MỐC 25% (PAIRED T-TEST 5-FOLD & MCNEMAR OOF TEST) ===")
display(stat_tests_df)
print("\n=== TỔNG HỢP KIỂM ĐỊNH THỐNG KÊ QUA 4 MỐC (25% -> 100%) ===")
display(stat_tests_4snaps_df)

2026-11-19 16:33:06,974 [INFO] Đã nạp snapshot 25%: 32,593 x 113


2026-11-19 16:33:07,374 [INFO] Đã nạp snapshot 50%: 32,593 x 113


2026-11-19 16:33:07,768 [INFO] Đã nạp snapshot 75%: 32,593 x 113


2026-11-19 16:33:08,177 [INFO] Đã nạp snapshot 100%: 32,593 x 113


2026-11-19 16:33:08,287 [INFO] Đã nạp kết quả Baseline 5-Fold CV từ week7_baseline_cv_metrics.json


2026-11-19 16:35:51,515 [INFO] Hoàn tất 5-Fold CV 5 Mô hình Nâng cao cho (25%, dynamic_active) trong 163.2s


... (2 outputs left out)


=== BẢNG ĐỐI SÁNH TOÀN DIỆN 10 MÔ HÌNH (5 BASELINE + 5 ADVANCED) — 5-FOLD CV TẠI MỐC 25% DYNAMIC ACTIVE ===


,family,model,accuracy_mean,precision_mean,recall_mean,recall_sd,f1_mean,f1_sd,roc_auc_mean,roc_auc_sd,pr_auc_mean,brier_mean
9,Advanced (7B),Ensemble / Stacking,0.7718,0.7225,0.7287,0.0084,0.7255,0.0060,0.8491,0.0032,0.8291,0.1564
7,Advanced (7B),CatBoost / HistGBM,0.7740,0.7330,0.7144,0.0074,0.7235,0.0065,0.8484,0.0031,0.8278,0.1547
5,Advanced (7B),XGBoost,0.7750,0.7376,0.7084,0.0084,0.7226,0.0056,0.8472,0.0032,0.8280,0.1541
6,Advanced (7B),LightGBM,0.7725,0.7342,0.7061,0.0081,0.7198,0.0059,0.8466,0.0028,0.8273,0.1546
0,Baseline (7A),Logistic Regression,0.7687,0.7254,0.7099,0.0112,0.7175,0.0067,0.8423,0.0027,0.8212,0.1577
2,Baseline (7A),Random Forest,0.7733,0.7524,0.6745,0.0105,0.7113,0.0073,0.8416,0.0040,0.8204,0.1569
8,Advanced (7B),Neural Network (MLP),0.7751,0.7804,0.6355,0.0138,0.7005,0.0099,0.8401,0.0041,0.8198,0.1552
3,Baseline (7A),SVM,0.7459,0.8772,0.4491,0.0221,0.5937,0.0186,0.8206,0.0074,0.7989,0.1799
1,Baseline (7A),Decision Tree,0.7373,0.6897,0.6694,0.0398,0.6781,0.0102,0.8093,0.0063,0.7783,0.1740
4,Baseline (7A),KNN,0.7534,0.8050,0.5334,0.0121,0.6415,0.0081,0.8071,0.0048,0.7865,0.1707



=== KIỂM ĐỊNH THỐNG KÊ (PAIRED T-TEST 5-FOLD & MCNEMAR OOF TEST) ===


,advanced_model,baseline_model,delta_auc_mean,paired_t_auc_p,delta_f1_mean,paired_t_f1_p,mcnemar_chi2,mcnemar_p
0,LightGBM,Random Forest,0.0050,0.0016,0.0086,0.0113,0.2598,0.6103
1,LightGBM,Logistic Regression,0.0043,0.0073,0.0023,0.3832,4.2724,0.0387
2,XGBoost,Random Forest,0.0055,0.0006,0.0114,0.0013,0.8728,0.3502
3,XGBoost,Logistic Regression,0.0049,0.0087,0.0051,0.0671,11.9697,0.0005
4,CatBoost / HistGBM,Random Forest,0.0067,0.0004,0.0123,0.0002,0.1793,0.6720
5,CatBoost / HistGBM,Logistic Regression,0.0060,0.0108,0.0060,0.0026,10.1292,0.0015
6,Ensemble / Stacking,Random Forest,0.0075,0.0000,0.0143,0.0003,0.9902,0.3197
7,Ensemble / Stacking,Logistic Regression,0.0068,0.0045,0.0080,0.0186,3.3738,0.0662


## 3. Hyperparameter Tuning (Nested 5-Fold $\times$ 3-Fold Group CV) & Tối Ưu Ngưỡng In-Fold $\tau^*_k$
Thực hiện tinh chỉnh siêu tham số **hoàn toàn lồng trong từng Outer Fold** (`RandomizedSearchCV` với `StratifiedGroupKFold(n_splits=3)` trên 4 phần Train) kết hợp tách nội bộ $85\%$ Inner-Train / $15\%$ Inner-Val theo `id_student` để học ngưỡng quyết định $\tau^*_k$ ($\text{Recall}^{\text{inner-val}} \ge 0.76$), đảm bảo **0% rò rỉ siêu tham số và ngưỡng từ phần Test**.

In [4]:
tuning_results: List[Dict[str, Any]] = []
tuned_fold_configs: Dict[str, List[Dict[str, Any]]] = {"LightGBM (Tuned)": [], "XGBoost (Tuned)": [], "CatBoost / HistGBM (Tuned)": []}
tuned_oof_probs: Dict[str, np.ndarray] = {}

splits_25 = engine.get_5fold_splits(df_25_dyn, n_splits=5, random_state=42)
y_25 = df_25_dyn["at_risk"].values
groups_25 = df_25_dyn["id_student"].values

param_spaces = {
    "LightGBM (Tuned)": {
        "n_estimators": [150, 220, 300],
        "num_leaves": [20, 31, 45],
        "max_depth": [5, 7, -1],
        "learning_rate": [0.03, 0.05, 0.08],
        "min_child_samples": [20, 35, 60],
        "subsample": [0.8, 0.9, 1.0],
        "colsample_bytree": [0.75, 0.85, 1.0],
        "reg_lambda": [0.0, 1.0, 5.0],
    },
    "XGBoost (Tuned)": {
        "n_estimators": [150, 220, 300],
        "max_depth": [4, 5, 6],
        "learning_rate": [0.03, 0.05, 0.07],
        "min_child_weight": [1, 3, 5, 10],
        "subsample": [0.8, 0.9, 1.0],
        "colsample_bytree": [0.75, 0.85, 1.0],
        "reg_lambda": [1.0, 3.0, 5.0],
    },
    "CatBoost / HistGBM (Tuned)": (
        {
            "iterations": [150, 220],
            "depth": [5, 6],
            "learning_rate": [0.04, 0.06],
            "l2_leaf_reg": [2.0, 5.0],
            "rsm": [0.8],
        }
        if CATBOOST_AVAILABLE
        else {
            "max_iter": [150, 220, 300],
            "max_depth": [5, 6, 8],
            "learning_rate": [0.03, 0.05, 0.07],
            "min_samples_leaf": [20, 35, 60],
            "l2_regularization": [0.1, 1.0, 5.0],
        }
    ),
}

def make_base_tunable(name: str, pos_w: float) -> Any:
    if name == "LightGBM (Tuned)":
        return LGBMClassifier(class_weight="balanced", random_state=42, n_jobs=-1, verbose=-1)
    elif name == "XGBoost (Tuned)":
        return XGBClassifier(scale_pos_weight=pos_w, eval_metric="logloss", random_state=42, n_jobs=-1)
    else:
        if CATBOOST_AVAILABLE:
            return CatBoostClassifier(auto_class_weights="Balanced", random_seed=42, verbose=0, thread_count=-1)
        return HistGradientBoostingClassifier(class_weight="balanced", random_state=42)

for m_name in ["LightGBM (Tuned)", "XGBoost (Tuned)", "CatBoost / HistGBM (Tuned)"]:
    t0_m = time.time()
    oof_p = np.zeros(len(df_25_dyn), dtype=float)
    oof_pred_tuned = np.zeros(len(df_25_dyn), dtype=int)
    fold_rows = []

    for fold, tr_idx, te_idx in splits_25:
        tr_df_f = df_25_dyn.iloc[tr_idx].reset_index(drop=True)
        te_df_f = df_25_dyn.iloc[te_idx].reset_index(drop=True)
        X_tr, X_te, _ = engine.prepare_fold_matrices(tr_df_f, te_df_f)
        y_tr, y_te = y_25[tr_idx], y_25[te_idx]
        g_tr = groups_25[tr_idx]
        pos_w = float((y_tr == 0).sum() / max(1, (y_tr == 1).sum()))

        # 1. Nested RandomizedSearchCV bên trong 4 phần Train (inner 3-fold StratifiedGroupKFold)
        inner_cv = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=42 + fold)
        search = RandomizedSearchCV(
            estimator=make_base_tunable(m_name, pos_w),
            param_distributions=param_spaces[m_name],
            n_iter=5,
            scoring="roc_auc",
            cv=inner_cv,
            random_state=42 + fold,
            n_jobs=1,
            refit=True,
        )
        search.fit(X_tr, y_tr, groups=g_tr)
        best_params_k = search.best_params_

        # 2. Tìm ngưỡng tối ưu tau*_k trên 15% Inner-Val (tách theo id_student bên trong 4 phần Train)
        inner_split = StratifiedGroupKFold(n_splits=6, shuffle=True, random_state=100 + fold)
        in_tr_idx, in_va_idx = next(inner_split.split(tr_df_f, y_tr, groups=g_tr))
        X_in_tr, X_in_va, _ = engine.prepare_fold_matrices(tr_df_f.iloc[in_tr_idx], tr_df_f.iloc[in_va_idx])
        y_in_tr, y_in_va = y_tr[in_tr_idx], y_tr[in_va_idx]
        pos_w_in = float((y_in_tr == 0).sum() / max(1, (y_in_tr == 1).sum()))

        clf_inner = make_base_tunable(m_name, pos_w_in).set_params(**best_params_k)
        clf_inner.fit(X_in_tr, y_in_tr)
        prob_in_va = clf_inner.predict_proba(X_in_va)[:, 1]
        tau_k, inner_rec, inner_f1 = tune_threshold_inner_val(y_in_va, prob_in_va, min_recall_target=0.76)

        # 3. Đánh giá 1 lần duy nhất lên phần Test thứ k với mô hình đã refit trên toàn bộ 4 phần Train
        best_est_k = search.best_estimator_
        prob_te = best_est_k.predict_proba(X_te)[:, 1]
        oof_p[te_idx] = prob_te
        oof_pred_tuned[te_idx] = (prob_te >= tau_k).astype(int)

        m_default = engine.evaluate_predictions(y_te, prob_te, threshold=0.50)
        m_tuned = engine.evaluate_predictions(y_te, prob_te, threshold=tau_k)

        fold_rows.append({
            "model": m_name,
            "fold": fold,
            "tau_k": tau_k,
            "inner_cv_roc_auc": round(float(search.best_score_), 4),
            "default_recall": round(m_default["recall"], 4),
            "tuned_recall": round(m_tuned["recall"], 4),
            "tuned_precision": round(m_tuned["precision"], 4),
            "tuned_f1": round(m_tuned["f1"], 4),
            "tuned_accuracy": round(m_tuned["accuracy"], 4),
            "roc_auc": round(m_tuned["roc_auc"], 4),
            "pr_auc": round(m_tuned["pr_auc"], 4),
            "brier": round(m_tuned["brier"], 4),
            "best_params": best_params_k,
        })
        tuned_fold_configs[m_name].append({"fold": fold, "tau_k": tau_k, "params": best_params_k, "roc_auc": m_tuned["roc_auc"], "recall": m_tuned["recall"], "f1": m_tuned["f1"]})

    tuned_oof_probs[m_name] = oof_p
    fdf = pd.DataFrame(fold_rows)
    tuning_results.extend(fold_rows)
    logger.info(
        f"[{m_name}] 5-Fold Nested Tuning xong ({time.time() - t0_m:.1f}s) -> "
        f"Recall={fdf['tuned_recall'].mean():.4f}±{fdf['tuned_recall'].std():.4f}, "
        f"F1={fdf['tuned_f1'].mean():.4f}±{fdf['tuned_f1'].std():.4f}, "
        f"AUC={fdf['roc_auc'].mean():.4f}±{fdf['roc_auc'].std():.4f}"
    )

tuning_df = pd.DataFrame(tuning_results)
tuning_summary = tuning_df.groupby("model").agg(
    tau_mean=("tau_k", "mean"),
    default_recall_mean=("default_recall", "mean"),
    tuned_recall_mean=("tuned_recall", "mean"),
    tuned_recall_sd=("tuned_recall", "std"),
    tuned_precision_mean=("tuned_precision", "mean"),
    tuned_f1_mean=("tuned_f1", "mean"),
    tuned_f1_sd=("tuned_f1", "std"),
    roc_auc_mean=("roc_auc", "mean"),
    roc_auc_sd=("roc_auc", "std"),
    pr_auc_mean=("pr_auc", "mean"),
    brier_mean=("brier", "mean"),
).reset_index()

print("\n=== KẾT QUẢ NESTED 5-FOLD TUNING & TỐI ƯU NGƯỠNG IN-FOLD (25% DYNAMIC ACTIVE) ===")
display(tuning_summary)

2026-11-19 16:43:40,932 [INFO] [LightGBM (Tuned)] 5-Fold Nested Tuning xong (103.3s) -> Recall=0.7867±0.0186, F1=0.7252±0.0054, AUC=0.8472±0.0035


2026-11-19 16:45:40,350 [INFO] [XGBoost (Tuned)] 5-Fold Nested Tuning xong (119.4s) -> Recall=0.7890±0.0149, F1=0.7250±0.0065, AUC=0.8485±0.0027


2026-11-19 16:49:56,258 [INFO] [CatBoost / HistGBM (Tuned)] 5-Fold Nested Tuning xong (255.9s) -> Recall=0.7836±0.0120, F1=0.7252±0.0075, AUC=0.8479±0.0036



=== KẾT QUẢ NESTED 5-FOLD TUNING & TỐI ƯU NGƯỠNG IN-FOLD (25% DYNAMIC ACTIVE) ===


,model,tau_mean,default_recall_mean,tuned_recall_mean,tuned_recall_sd,tuned_precision_mean,tuned_f1_mean,tuned_f1_sd,roc_auc_mean,roc_auc_sd,pr_auc_mean,brier_mean
0,CatBoost / HistGBM (Tuned),0.4180,0.7160,0.7836,0.0120,0.6754,0.7252,0.0075,0.8479,0.0036,0.8269,0.1552
1,LightGBM (Tuned),0.4140,0.7131,0.7867,0.0186,0.6731,0.7252,0.0054,0.8472,0.0035,0.8276,0.1550
2,XGBoost (Tuned),0.4080,0.7131,0.7890,0.0149,0.6710,0.7250,0.0065,0.8485,0.0027,0.8291,0.1541


## 4. Thực Nghiệm Dự Đoán Sớm (Early Prediction) Qua 4 Mốc `25%`, `50%`, `75%`, `100%` (5-Fold CV)
Đánh giá bộ mô hình đã tinh chỉnh (`LightGBM (Tuned)`, `XGBoost (Tuned)`, `CatBoost / HistGBM (Tuned)`, `Ensemble / Stacking (Tuned)`) qua tất cả 4 mốc cắt thời gian trên cả `dynamic_active` và `full` dưới quy chuẩn 5-Fold CV + In-Fold $\tau^*_k$.

In [ ]:
# Chọn tham số đồng thuận (consensus best params từ Nested Tuning) cho đánh giá đa lát cắt nhanh và ổn định
def get_consensus_params(fold_list: List[Dict[str, Any]]) -> Dict[str, Any]:
    # Lấy bộ tham số từ fold đạt AUC cao nhất làm đại diện tinh chỉnh
    best_entry = max(fold_list, key=lambda x: x["roc_auc"])
    return dict(best_entry["params"])

consensus_lgbm_params = get_consensus_params(tuned_fold_configs["LightGBM (Tuned)"])
consensus_xgb_params = get_consensus_params(tuned_fold_configs["XGBoost (Tuned)"])
consensus_hgb_params = get_consensus_params(tuned_fold_configs["CatBoost / HistGBM (Tuned)"])

print("Consensus Tuned Params - LightGBM:", consensus_lgbm_params)
print("Consensus Tuned Params - XGBoost :", consensus_xgb_params)
print("Consensus Tuned Params - Cat/HGB :", consensus_hgb_params)

def build_tuned_models_dict(pos_w: float, fast_inner: bool = False) -> Dict[str, Any]:
    lgbm_t = LGBMClassifier(class_weight="balanced", random_state=42, n_jobs=-1, verbose=-1, **consensus_lgbm_params)
    xgb_t = XGBClassifier(scale_pos_weight=pos_w, eval_metric="logloss", random_state=42, n_jobs=-1, **consensus_xgb_params)
    if CATBOOST_AVAILABLE:
        hgb_t = CatBoostClassifier(auto_class_weights="Balanced", random_seed=42, verbose=0, thread_count=-1, **consensus_hgb_params)
    else:
        hgb_t = HistGradientBoostingClassifier(class_weight="balanced", random_state=42, **consensus_hgb_params)

    stack_t = StackingClassifier(
        estimators=[
            ("lgbm", clone(lgbm_t)),
            ("xgb", clone(xgb_t)),
            ("hgb", clone(hgb_t)),
            ("rf", RandomForestClassifier(class_weight="balanced", n_estimators=150, max_depth=12, min_samples_leaf=5, n_jobs=-1, random_state=42)),
        ],
        final_estimator=LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42),
        cv=3,
        n_jobs=1,
    )
    return {
        "LightGBM (Tuned)": lgbm_t,
        "XGBoost (Tuned)": xgb_t,
        "CatBoost / HistGBM (Tuned)": hgb_t,
        "Ensemble / Stacking (Tuned)": stack_t,
    }

early_per_fold_rows: List[Dict[str, Any]] = []
early_summary_rows: List[Dict[str, Any]] = []
oof_early_store: Dict[str, Dict[str, Any]] = {}

for snap_label in ["25%", "50%", "75%", "100%"]:
    for cohort_name in ["dynamic_active", "full"]:
        t0_e = time.time()
        df_c = engine.filter_cohort(snapshots[snap_label], cohort_name)
        y_all = df_c["at_risk"].values
        g_all = df_c["id_student"].values
        splits = engine.get_5fold_splits(df_c, n_splits=5, random_state=42)

        eval_models = ["LightGBM (Tuned)", "XGBoost (Tuned)", "CatBoost / HistGBM (Tuned)"]
        if cohort_name == "dynamic_active":
            eval_models.append("Ensemble / Stacking (Tuned)")

        oof_probs = {m: np.zeros(len(df_c), dtype=float) for m in eval_models}
        oof_preds = {m: np.zeros(len(df_c), dtype=int) for m in eval_models}
        fold_records = {m: [] for m in eval_models}

        for fold, tr_idx, te_idx in splits:
            tr_df_f = df_c.iloc[tr_idx].reset_index(drop=True)
            te_df_f = df_c.iloc[te_idx].reset_index(drop=True)
            X_tr, X_te, _ = engine.prepare_fold_matrices(tr_df_f, te_df_f)
            y_tr, y_te = y_all[tr_idx], y_all[te_idx]
            g_tr = g_all[tr_idx]
            pos_w = float((y_tr == 0).sum() / max(1, (y_tr == 1).sum()))

            in_sgkf = StratifiedGroupKFold(n_splits=6, shuffle=True, random_state=200 + fold)
            in_tr_idx, in_va_idx = next(in_sgkf.split(tr_df_f, y_tr, groups=g_tr))
            X_in_tr, X_in_va, _ = engine.prepare_fold_matrices(tr_df_f.iloc[in_tr_idx], tr_df_f.iloc[in_va_idx])
            y_in_tr, y_in_va = y_tr[in_tr_idx], y_tr[in_va_idx]
            pos_w_in = float((y_in_tr == 0).sum() / max(1, (y_in_tr == 1).sum()))

            in_models = build_tuned_models_dict(pos_w_in)
            full_models = build_tuned_models_dict(pos_w)

            for m_name in eval_models:
                clf_in = in_models[m_name]
                clf_in.fit(X_in_tr, y_in_tr)
                tau_k, _, _ = tune_threshold_inner_val(y_in_va, clf_in.predict_proba(X_in_va)[:, 1], min_recall_target=0.76)

                clf_full = full_models[m_name]
                clf_full.fit(X_tr, y_tr)
                prob_te = clf_full.predict_proba(X_te)[:, 1]
                pred_te = (prob_te >= tau_k).astype(int)

                oof_probs[m_name][te_idx] = prob_te
                oof_preds[m_name][te_idx] = pred_te

                m_def = engine.evaluate_predictions(y_te, prob_te, threshold=0.50)
                m_opt = engine.evaluate_predictions(y_te, prob_te, threshold=tau_k)
                row = {
                    "snapshot": snap_label,
                    "cohort": cohort_name,
                    "n_cohort": int(len(df_c)),
                    "model": m_name,
                    "fold": int(fold),
                    "tau_k": float(tau_k),
                    "default_recall": round(m_def["recall"], 4),
                    "default_f1": round(m_def["f1"], 4),
                    **{k: round(m_opt[k], 4) for k in ["accuracy", "precision", "recall", "f1", "roc_auc", "pr_auc", "balanced_accuracy", "brier"]},
                }
                early_per_fold_rows.append(row)
                fold_records[m_name].append(row)

        for m_name in eval_models:
            fdf = pd.DataFrame(fold_records[m_name])
            oof_m = engine.evaluate_predictions(y_all, oof_probs[m_name], threshold=0.50)
            oof_rec_exact = float(recall_score(y_all, oof_preds[m_name], zero_division=0))
            oof_f1_exact = float(f1_score(y_all, oof_preds[m_name], zero_division=0))
            tn, fp, fn, tp = confusion_matrix(y_all, oof_preds[m_name], labels=[0, 1]).ravel()
            s_row = {
                "snapshot": snap_label,
                "cohort": cohort_name,
                "n_cohort": int(len(df_c)),
                "at_risk_rate": round(float(y_all.mean()), 4),
                "model": m_name,
                "tau_mean": round(float(fdf["tau_k"].mean()), 4),
                "tau_sd": round(float(fdf["tau_k"].std(ddof=1)), 4),
                "default_recall_mean": round(float(fdf["default_recall"].mean()), 4),
            }
            for k in ["accuracy", "precision", "recall", "f1", "roc_auc", "pr_auc"]:
                s_row[f"{k}_mean"] = round(float(fdf[k].mean()), 4)
                s_row[f"{k}_sd"] = round(float(fdf[k].std(ddof=1)), 4)
            s_row["brier_mean"] = round(float(fdf["brier"].mean()), 4)
            s_row["oof_recall_exact"] = round(oof_rec_exact, 4)
            s_row["oof_f1_exact"] = round(oof_f1_exact, 4)
            s_row["oof_roc_auc"] = round(oof_m["roc_auc"], 4)
            s_row["oof_pr_auc"] = round(oof_m["pr_auc"], 4)
            s_row["oof_tn"] = int(tn)
            s_row["oof_fp"] = int(fp)
            s_row["oof_fn"] = int(fn)
            s_row["oof_tp"] = int(tp)
            early_summary_rows.append(s_row)

        if cohort_name == "dynamic_active" and snap_label in {"25%", "100%"}:
            oof_early_store[f"{snap_label}_{cohort_name}"] = {
                "y_true": y_all.astype(int).tolist(),
                "probs": {m: np.round(oof_probs[m], 5).tolist() for m in eval_models},
                "preds": {m: oof_preds[m].astype(int).tolist() for m in eval_models},
            }

        logger.info(f"Hoàn tất Early Prediction 5-Fold ({snap_label}, {cohort_name}) trong {time.time() - t0_e:.1f}s")

early_summary_df = pd.DataFrame(early_summary_rows)
print("\n=== TỔNG HỢP EARLY PREDICTION 5-FOLD CV (DYNAMIC ACTIVE: 25% -> 100%) ===")
display(early_summary_df[early_summary_df["cohort"] == "dynamic_active"][["snapshot", "model", "tau_mean", "recall_mean", "recall_sd", "f1_mean", "f1_sd", "roc_auc_mean", "pr_auc_mean"]])

## 5. Thử Nghiệm Mở Rộng Phân Loại Đa Lớp 4-Class (`Distinction`, `Pass`, `Fail`, `Withdrawn`) & Cầu Nối Xác Suất Gộp
Đánh giá 5-Fold CV cho bài toán phân loại đa lớp gốc 4 nhãn bằng `LightGBM` và `XGBoost` tại mốc sớm **25%** và toàn khóa **100%** (`dynamic_active`), đồng thời kiểm chứng cầu nối xác suất gộp $P(\text{At-Risk}) = P(\text{Fail}) + P(\text{Withdrawn})$.

In [6]:
multiclass_fold_rows: List[Dict[str, Any]] = []
multiclass_summary_rows: List[Dict[str, Any]] = []
multiclass_cm_store: Dict[str, List[List[int]]] = {}
class_labels_4 = ["Distinction", "Pass", "Fail", "Withdrawn"]

for snap_label in ["25%", "100%"]:
    df_c = engine.filter_cohort(snapshots[snap_label], "dynamic_active")
    y4_all = df_c["final_result"].map(engine.CLASS_4_MAP).values
    y2_all = df_c["at_risk"].values
    splits = engine.get_5fold_splits(df_c, n_splits=5, random_state=42)

    for m_name in ["LightGBM (4-Class)", "XGBoost (4-Class)"]:
        oof_probs_4 = np.zeros((len(df_c), 4), dtype=float)
        fold_rows_4 = []

        for fold, tr_idx, te_idx in splits:
            X_tr, X_te, _ = engine.prepare_fold_matrices(df_c.iloc[tr_idx], df_c.iloc[te_idx])
            y4_tr, y4_te = y4_all[tr_idx], y4_all[te_idx]

            if m_name.startswith("LightGBM"):
                lgb4_params = {**consensus_lgbm_params, "n_estimators": 220}
                clf4 = LGBMClassifier(
                    objective="multiclass", num_class=4, class_weight="balanced",
                    random_state=42, n_jobs=-1, verbose=-1, **lgb4_params,
                )
            else:
                xgb4_params = {**consensus_xgb_params, "n_estimators": 220}
                clf4 = XGBClassifier(
                    objective="multi:softprob", num_class=4, eval_metric="mlogloss",
                    random_state=42, n_jobs=-1, **xgb4_params,
                )

            clf4.fit(X_tr, y4_tr)
            prob4_te = clf4.predict_proba(X_te)
            pred4_te = np.argmax(prob4_te, axis=1)
            oof_probs_4[te_idx] = prob4_te

            # Cầu nối xác suất gộp At-Risk = P(Fail [2]) + P(Withdrawn [3])
            prob_at_risk_collapsed = prob4_te[:, 2] + prob4_te[:, 3]
            auc_collapsed = float(roc_auc_score(y2_all[te_idx], prob_at_risk_collapsed))

            row4 = {
                "snapshot": snap_label,
                "model": m_name,
                "fold": int(fold),
                "accuracy_4class": round(float(accuracy_score(y4_te, pred4_te)), 4),
                "macro_f1_4class": round(float(f1_score(y4_te, pred4_te, average="macro")), 4),
                "weighted_f1_4class": round(float(f1_score(y4_te, pred4_te, average="weighted")), 4),
                "ovr_roc_auc_4class": round(float(roc_auc_score(y4_te, prob4_te, multi_class="ovr", average="macro")), 4),
                "collapsed_binary_roc_auc": round(auc_collapsed, 4),
            }
            fold_rows_4.append(row4)
            multiclass_fold_rows.append(row4)

        f4_df = pd.DataFrame(fold_rows_4)
        oof_pred_4 = np.argmax(oof_probs_4, axis=1)
        cm4 = confusion_matrix(y4_all, oof_pred_4, labels=[0, 1, 2, 3])
        multiclass_cm_store[f"{snap_label}_{m_name}"] = cm4.tolist()

        # Kiểm tra McNemar giữa dự đoán gộp 4-class vs Binary Tuned model ở ngưỡng mặc định 0.50
        p_collapsed_all = oof_probs_4[:, 2] + oof_probs_4[:, 3]
        multiclass_summary_rows.append({
            "snapshot": snap_label,
            "model": m_name,
            "accuracy_4class_mean": round(float(f4_df["accuracy_4class"].mean()), 4),
            "accuracy_4class_sd": round(float(f4_df["accuracy_4class"].std(ddof=1)), 4),
            "macro_f1_4class_mean": round(float(f4_df["macro_f1_4class"].mean()), 4),
            "macro_f1_4class_sd": round(float(f4_df["macro_f1_4class"].std(ddof=1)), 4),
            "weighted_f1_4class_mean": round(float(f4_df["weighted_f1_4class"].mean()), 4),
            "ovr_roc_auc_mean": round(float(f4_df["ovr_roc_auc_4class"].mean()), 4),
            "collapsed_binary_roc_auc_mean": round(float(f4_df["collapsed_binary_roc_auc"].mean()), 4),
            "collapsed_binary_recall_at_05": round(float(recall_score(y2_all, (p_collapsed_all >= 0.50).astype(int))), 4),
        })

multiclass_summary_df = pd.DataFrame(multiclass_summary_rows)
print("=== KẾT QUẢ PHÂN LOẠI ĐA LỚP 4-CLASS (5-FOLD CV, DYNAMIC ACTIVE) ===")
display(multiclass_summary_df)

=== KẾT QUẢ PHÂN LOẠI ĐA LỚP 4-CLASS (5-FOLD CV, DYNAMIC ACTIVE) ===


,snapshot,model,accuracy_4class_mean,accuracy_4class_sd,macro_f1_4class_mean,macro_f1_4class_sd,weighted_f1_4class_mean,ovr_roc_auc_mean,collapsed_binary_roc_auc_mean,collapsed_binary_recall_at_05
0,25%,LightGBM (4-Class),0.5135,0.0072,0.4857,0.0059,0.5214,0.7912,0.8464,0.7752
1,25%,XGBoost (4-Class),0.6000,0.0091,0.4605,0.0104,0.5583,0.7947,0.8483,0.6336
2,100%,LightGBM (4-Class),0.7753,0.0104,0.6851,0.0122,0.7841,0.9402,0.9646,0.8404
3,100%,XGBoost (4-Class),0.8203,0.0103,0.6650,0.0222,0.8153,0.9436,0.9657,0.8054


## 6. Lưu Trữ Mô Hình Tốt Nhất (`models/`), Tổng Hợp Kết Quả (`data/processed/week7_modeling/`) & Xuất Biểu Đồ (Hình 34–37)
Đóng gói mô hình thắng cuộc ở bài toán Cảnh báo sớm `25% dynamic_active` kèm bộ 5 mô hình & 5 bộ tiền xử lý của 5 Fold để phục vụ XAI (Tuần 8A) và Demo Streamlit (Tuần 8B).

In [4]:
# 1. Xác định mô hình tốt nhất tại mốc cảnh báo sớm trọng tâm 25% dynamic_active
sub_early_25 = early_summary_df[
    (early_summary_df["snapshot"] == "25%") & (early_summary_df["cohort"] == "dynamic_active")
].sort_values(["roc_auc_mean", "f1_mean", "recall_mean"], ascending=False)
display(sub_early_25[["model", "tau_mean", "recall_mean", "recall_sd", "f1_mean", "f1_sd", "roc_auc_mean", "roc_auc_sd", "pr_auc_mean"]])

# Chọn Best Tree Model trực tiếp (tương thích hoàn toàn với TreeExplainer / SHAP / feature_importances_ ở Tuần 8)
# Đồng thời lưu cả Best Ensemble nếu Ensemble dẫn đầu AUC-ROC
best_overall_row = sub_early_25.iloc[0]
best_tree_candidates = sub_early_25[sub_early_25["model"].isin(["LightGBM (Tuned)", "XGBoost (Tuned)"])]
best_tree_row = best_tree_candidates.iloc[0]
best_tree_name = str(best_tree_row["model"])
best_tree_tau = float(best_tree_row["tau_mean"])

logger.info(f"Best Overall 5-Fold Model (25% Dynamic): {best_overall_row['model']} (AUC={best_overall_row['roc_auc_mean']:.4f}, F1={best_overall_row['f1_mean']:.4f}, Rec={best_overall_row['recall_mean']:.4f})")
logger.info(f"Best Interpretable Tree Model (models/best_model.pkl): {best_tree_name} (AUC={best_tree_row['roc_auc_mean']:.4f}, F1={best_tree_row['f1_mean']:.4f}, Rec={best_tree_row['recall_mean']:.4f}, tau*={best_tree_tau:.3f})")

# Huấn luyện 5-Fold Bundle và Full-Fit Best Tree Model trên 25% dynamic_active
df_25_dyn = engine.filter_cohort(snapshots["25%"], "dynamic_active")
y_25 = df_25_dyn["at_risk"].values
g_25 = df_25_dyn["id_student"].values
splits_25 = engine.get_5fold_splits(df_25_dyn, n_splits=5, random_state=42)

fold_models = []
fold_preprocessors = []
fold_taus = []

for fold, tr_idx, te_idx in splits_25:
    tr_df_f = df_25_dyn.iloc[tr_idx].reset_index(drop=True)
    X_tr, _, state_k = engine.prepare_fold_matrices(tr_df_f, df_25_dyn.iloc[te_idx])
    y_tr = y_25[tr_idx]
    g_tr = g_25[tr_idx]
    pos_w = float((y_tr == 0).sum() / max(1, (y_tr == 1).sum()))

    in_sgkf = StratifiedGroupKFold(n_splits=6, shuffle=True, random_state=200 + fold)
    in_tr_idx, in_va_idx = next(in_sgkf.split(tr_df_f, y_tr, groups=g_tr))
    X_in_tr, X_in_va, _ = engine.prepare_fold_matrices(tr_df_f.iloc[in_tr_idx], tr_df_f.iloc[in_va_idx])
    y_in_tr, y_in_va = y_tr[in_tr_idx], y_tr[in_va_idx]
    pos_w_in = float((y_in_tr == 0).sum() / max(1, (y_in_tr == 1).sum()))

    clf_in = build_tuned_models_dict(pos_w_in)[best_tree_name]
    clf_in.fit(X_in_tr, y_in_tr)
    tau_k, _, _ = tune_threshold_inner_val(y_in_va, clf_in.predict_proba(X_in_va)[:, 1], min_recall_target=0.76)

    clf_k = build_tuned_models_dict(pos_w)[best_tree_name]
    clf_k.fit(X_tr, y_tr)
    fold_models.append(clf_k)
    fold_preprocessors.append(state_k)
    fold_taus.append(float(tau_k))

# Fit 1 mô hình chuẩn trên toàn bộ 25% dynamic_active để lưu thành models/best_model.pkl (trực tiếp gọi .predict_proba và feature_importances_)
full_state = engine.fit_preprocessor(df_25_dyn)
X_full_25 = engine.transform_with_state(df_25_dyn, full_state)
pos_w_full = float((y_25 == 0).sum() / max(1, (y_25 == 1).sum()))
best_tree_full = build_tuned_models_dict(pos_w_full)[best_tree_name]
best_tree_full.fit(X_full_25, y_25)

best_model_pkl_path = MODELS_DIR / "best_model.pkl"
joblib.dump(best_tree_full, best_model_pkl_path)

ensemble_bundle_path = MODELS_DIR / "best_model_5fold_ensemble.joblib"
joblib.dump({
    "model_name": best_tree_name,
    "feature_names_114": list(X_full_25.columns),
    "fold_models": fold_models,
    "fold_preprocessors": fold_preprocessors,
    "fold_thresholds": fold_taus,
    "mean_threshold": float(np.mean(fold_taus)),
    "full_preprocessor_state": full_state,
}, ensemble_bundle_path)

best_meta_path = MODELS_DIR / "best_model_metadata.json"
best_metadata = {
    "best_overall_model": str(best_overall_row["model"]),
    "best_tree_model_saved": best_tree_name,
    "catboost_status": CATBOOST_STATUS_MSG,
    "snapshot_target": "25% dynamic_active",
    "n_cohort": int(len(df_25_dyn)),
    "n_features_encoded": int(X_full_25.shape[1]),
    "feature_names_114": list(X_full_25.columns),
    "consensus_params": consensus_lgbm_params if "LightGBM" in best_tree_name else consensus_xgb_params,
    "fold_thresholds": fold_taus,
    "mean_threshold": round(float(np.mean(fold_taus)), 4),
    "cv_5fold_performance_25pct_dynamic": {
        "roc_auc_mean": float(best_tree_row["roc_auc_mean"]),
        "roc_auc_sd": float(best_tree_row["roc_auc_sd"]),
        "pr_auc_mean": float(best_tree_row["pr_auc_mean"]),
        "recall_mean": float(best_tree_row["recall_mean"]),
        "recall_sd": float(best_tree_row["recall_sd"]),
        "f1_mean": float(best_tree_row["f1_mean"]),
        "f1_sd": float(best_tree_row["f1_sd"]),
        "precision_mean": float(best_tree_row["precision_mean"]),
        "accuracy_mean": float(best_tree_row["accuracy_mean"]),
        "brier_mean": float(best_tree_row["brier_mean"]),
    },
}
with open(best_meta_path, "w", encoding="utf-8") as f:
    json.dump(best_metadata, f, indent=2, ensure_ascii=False)

# Xuất bảng CSV tổng hợp và JSON toàn bộ thực nghiệm Tuần 7 tại data/processed/week7_modeling/
base_sum_copy = baseline_summary_df.copy()
base_sum_copy["stage"] = "7A_Baseline_DefaultTau"
adv_sum_copy = adv_summary_df.copy()
adv_sum_copy["stage"] = "7B_Advanced_DefaultTau"
early_sum_copy = early_summary_df.copy()
early_sum_copy["stage"] = "7B_EarlyWarning_TunedInFoldTau"

common_cols = [
    "stage", "snapshot", "cohort", "n_cohort", "at_risk_rate", "model",
    "accuracy_mean", "accuracy_sd", "precision_mean", "precision_sd",
    "recall_mean", "recall_sd", "f1_mean", "f1_sd",
    "roc_auc_mean", "roc_auc_sd", "pr_auc_mean", "pr_auc_sd", "brier_mean",
]
cv_summary_all = pd.concat([
    base_sum_copy[common_cols],
    adv_sum_copy[common_cols],
    early_sum_copy[common_cols],
], ignore_index=True)
cv_csv_path = WEEK7_PROCESSED_DIR / "week7_cv_results_summary.csv"
cv_summary_all.to_csv(cv_csv_path, index=False)

week7_payload = {
    "protocol": baseline_payload["protocol"],
    "catboost_status": CATBOOST_STATUS_MSG,
    "best_model_metadata": best_metadata,
    "combined_10_models_25pct_dynamic": combined_10_df.to_dict(orient="records"),
    "combined_10_models_4snapshots_dynamic": combined_10_4snaps_df.to_dict(orient="records"),
    "statistical_tests_25pct_dynamic": stat_tests,
    "statistical_tests_4snapshots_dynamic": stat_tests_4snaps,
    "advanced_per_fold_results": adv_per_fold_records,
    "advanced_summary_results": adv_summary_records,
    "nested_tuning_per_fold": tuning_results,
    "nested_tuning_summary": tuning_summary.to_dict(orient="records"),
    "early_prediction_per_fold": early_per_fold_rows,
    "early_prediction_summary": early_summary_rows,
    "multiclass_4class_per_fold": multiclass_fold_rows,
    "multiclass_4class_summary": multiclass_summary_rows,
    "multiclass_4class_oof_cm": multiclass_cm_store,
    "oof_early_predictions": oof_early_store,
}
week7_json_path = WEEK7_PROCESSED_DIR / "week7_experiment_metrics.json"
with open(week7_json_path, "w", encoding="utf-8") as f:
    json.dump(week7_payload, f, indent=2, ensure_ascii=False)

# =========================================================================
# HÌNH 34: 34_week7_baseline_vs_advanced_5fold_benchmark.png
# =========================================================================
logger.info("Đang vẽ Hình 34: 34_week7_baseline_vs_advanced_5fold_benchmark.png")
fig, axes = plt.subplots(1, 2, figsize=(18.0, 6.8))

plot_10 = combined_10_df.sort_values("roc_auc_mean", ascending=True).reset_index(drop=True)
y_10 = np.arange(len(plot_10))
bar_colors = ["#95a5a6" if fam.startswith("Baseline") else "#2980b9" for fam in plot_10["family"]]

axes[0].barh(y_10, plot_10["roc_auc_mean"], xerr=plot_10["roc_auc_sd"], color=bar_colors, edgecolor="black", height=0.62, capsize=3)
axes[0].axvline(0.80, color="#c0392b", linestyle="--", linewidth=1.6, label="Mục tiêu Đề cương AUC = 0.80")
axes[0].set_yticks(y_10)
axes[0].set_yticklabels(plot_10["model"], fontsize=9.8, fontweight="bold")
axes[0].set_xlim(0.72, 0.93)
axes[0].set_xlabel("5-Fold Test AUC-ROC (Mean ± SD)", fontsize=10.5, fontweight="bold")
axes[0].set_title("(A) Đối sánh 10 Mô hình (5 Baseline vs 5 Advanced) — AUC-ROC", fontsize=11.8, fontweight="bold")
for yp, m_v, sd_v in zip(y_10, plot_10["roc_auc_mean"], plot_10["roc_auc_sd"]):
    axes[0].text(m_v + sd_v + 0.004, yp, f"{m_v:.4f} ± {sd_v:.4f}", va="center", fontsize=8.4, fontweight="bold")
from matplotlib.patches import Patch
axes[0].legend(
    handles=[
        Patch(facecolor="#95a5a6", edgecolor="black", label="5 Baseline (7A)"),
        Patch(facecolor="#2980b9", edgecolor="black", label="5 Advanced (7B)"),
        plt.Line2D([0], [0], color="#c0392b", linestyle="--", linewidth=1.6, label="Mục tiêu AUC = 0.80"),
    ],
    loc="lower right",
    fontsize=8.6,
)
axes[0].grid(axis="x", alpha=0.3)

# Panel B: PR-AUC vs F1-Score của 10 mô hình (tau=0.50)
h = 0.34
axes[1].barh(y_10 + h / 2, plot_10["pr_auc_mean"], xerr=plot_10["pr_auc_sd"], height=h, color="#8e44ad", edgecolor="black", capsize=2.5, label="PR-AUC (5-Fold Mean ± SD)")
axes[1].barh(y_10 - h / 2, plot_10["f1_mean"], xerr=plot_10["f1_sd"], height=h, color="#27ae60", edgecolor="black", capsize=2.5, label="F1-Score (5-Fold Mean ± SD, tau=0.50)")
axes[1].axvline(0.70, color="#d35400", linestyle="--", linewidth=1.5, label="Mục tiêu Đề cương F1 = 0.70")
axes[1].set_yticks(y_10)
axes[1].set_yticklabels(["" for _ in y_10])
axes[1].set_xlim(0.55, 0.94)
axes[1].set_ylim(-1.6, len(plot_10) - 0.3)
axes[1].set_xlabel("Điểm số 5-Fold (Mean ± SD)", fontsize=10.5, fontweight="bold")
axes[1].set_title("(B) PR-AUC & F1-Score của 10 Mô hình tại Mốc 25% Dynamic Active", fontsize=11.8, fontweight="bold")
for yp, pr_v, pr_sd, f1_v, f1_sd in zip(y_10, plot_10["pr_auc_mean"], plot_10["pr_auc_sd"], plot_10["f1_mean"], plot_10["f1_sd"]):
    axes[1].text(pr_v + pr_sd + 0.006, yp + h / 2, f"{pr_v:.3f}", va="center", fontsize=8.0, fontweight="bold", color="#6c3483")
    axes[1].text(f1_v + f1_sd + 0.006, yp - h / 2, f"{f1_v:.3f}", va="center", fontsize=8.0, fontweight="bold", color="#1e8449")
axes[1].legend(loc="lower right", fontsize=8.5)
axes[1].grid(axis="x", alpha=0.3)

plt.tight_layout()
fig.savefig(FIGURES_DIR / "34_week7_baseline_vs_advanced_5fold_benchmark.png", dpi=300)
plt.close(fig)

# =========================================================================
# HÌNH 35: 35_week7_early_prediction_5fold_progression_25_50_75_100.png
# =========================================================================
logger.info("Đang vẽ Hình 35: 35_week7_early_prediction_5fold_progression_25_50_75_100.png")
fig, axes = plt.subplots(1, 2, figsize=(17.0, 6.4))
x_snaps = [25, 50, 75, 100]
dyn_ew = early_summary_df[early_summary_df["cohort"] == "dynamic_active"].copy()
dyn_ew["pct_num"] = dyn_ew["snapshot"].str.replace("%", "").astype(int)

# Panel A: AUC-ROC Progression qua 4 mốc với Error Bars +-1 SD
model_styles = {
    "Ensemble / Stacking (Tuned)": ("#8e44ad", "D", "-"),
    "LightGBM (Tuned)": ("#2980b9", "o", "-"),
    "XGBoost (Tuned)": ("#27ae60", "s", "--"),
    "CatBoost / HistGBM (Tuned)": ("#e67e22", "^", "-."),
}
for m_name, (col, mk, ls) in model_styles.items():
    sub = dyn_ew[dyn_ew["model"] == m_name].sort_values("pct_num")
    axes[0].errorbar(
        x_snaps, sub["roc_auc_mean"], yerr=sub["roc_auc_sd"],
        color=col, marker=mk, linestyle=ls, linewidth=2.3, markersize=7, capsize=4, label=m_name
    )
    if m_name == "LightGBM (Tuned)":
        for xp, mv in zip(x_snaps, sub["roc_auc_mean"]):
            axes[0].text(xp, mv - 0.015, f"{mv:.4f}", ha="center", fontsize=8.5, fontweight="bold", color=col)

axes[0].axhline(0.80, color="#c0392b", linestyle=":", linewidth=1.6, label="Mục tiêu AUC = 0.80")
axes[0].set_xticks(x_snaps)
axes[0].set_xticklabels(["25%", "50%", "75%", "100%"])
axes[0].set_ylim(0.76, 0.99)
axes[0].set_xlabel("Mốc cắt Thời lượng Khóa học (%)", fontsize=10.5, fontweight="bold")
axes[0].set_ylabel("5-Fold Test AUC-ROC (Mean ± SD)", fontsize=10.5, fontweight="bold")
axes[0].set_title("(A) Tiến hóa AUC-ROC Early Prediction (5-Fold CV, Dynamic Active)", fontsize=11.5, fontweight="bold")
axes[0].legend(bbox_to_anchor=(0.98, 0.24), loc="lower right", fontsize=8.8)
axes[0].grid(alpha=0.3)

# Panel B: Recall & F1 Progression của Best Tree Model vs Full Cohort & Default Threshold
sub_best_dyn = dyn_ew[dyn_ew["model"] == best_tree_name].sort_values("pct_num")
full_ew = early_summary_df[(early_summary_df["cohort"] == "full") & (early_summary_df["model"] == best_tree_name)].copy()
full_ew["pct_num"] = full_ew["snapshot"].str.replace("%", "").astype(int)
full_ew = full_ew.sort_values("pct_num")

axes[1].plot(x_snaps, sub_best_dyn["default_recall_mean"], "--o", color="#95a5a6", linewidth=2.0, label=f"{best_tree_name} (tau=0.50) - Recall")
axes[1].errorbar(x_snaps, sub_best_dyn["recall_mean"], yerr=sub_best_dyn["recall_sd"], color="#c0392b", marker="D", linewidth=2.5, capsize=4, label=f"{best_tree_name} (In-Fold tau*) - Recall Dynamic")
axes[1].errorbar(x_snaps, sub_best_dyn["f1_mean"], yerr=sub_best_dyn["f1_sd"], color="#e67e22", marker="s", linewidth=2.2, capsize=4, label=f"{best_tree_name} (In-Fold tau*) - F1 Dynamic")
axes[1].plot(x_snaps, full_ew["recall_mean"], "-.^", color="#2ecc71", linewidth=2.0, label=f"{best_tree_name} (In-Fold tau*) - Recall Full Cohort")
axes[1].axhline(0.75, color="#196f3d", linestyle=":", linewidth=1.7, label="Ngưỡng Mục tiêu Recall >= 0.75")
axes[1].axhline(0.70, color="#117864", linestyle="-.", linewidth=1.4, label="Ngưỡng Mục tiêu F1 >= 0.70")

for xp, r_v, f_v in zip(x_snaps, sub_best_dyn["recall_mean"], sub_best_dyn["f1_mean"]):
    if r_v >= f_v:
        axes[1].text(xp, r_v + 0.017, f"{r_v:.3f}", ha="center", fontsize=8.5, fontweight="bold", color="#922b21")
        axes[1].text(xp, f_v - 0.025, f"{f_v:.3f}", ha="center", fontsize=8.5, fontweight="bold", color="#a04000")
    else:
        axes[1].text(xp, f_v + 0.017, f"{f_v:.3f}", ha="center", fontsize=8.5, fontweight="bold", color="#a04000")
        axes[1].text(xp, r_v - 0.026, f"{r_v:.3f}", ha="center", fontsize=8.5, fontweight="bold", color="#922b21")

axes[1].set_xticks(x_snaps)
axes[1].set_xticklabels(["25%", "50%", "75%", "100%"])
axes[1].set_ylim(0.55, 1.00)
axes[1].set_xlabel("Mốc cắt Thời lượng Khóa học (%)", fontsize=10.5, fontweight="bold")
axes[1].set_ylabel("5-Fold Test Recall / F1-Score (Mean ± SD)", fontsize=10.5, fontweight="bold")
axes[1].set_title("(B) Đột phá Recall & F1 Early Warning qua 4 Mốc (In-Fold Tuned tau*)", fontsize=11.5, fontweight="bold")
axes[1].legend(loc="lower right", fontsize=8.1)
axes[1].grid(alpha=0.3)

plt.tight_layout()
fig.savefig(FIGURES_DIR / "35_week7_early_prediction_5fold_progression_25_50_75_100.png", dpi=300)
plt.close(fig)

# =========================================================================
# HÌNH 36: 36_week7_hyperparameter_tuning_and_4class_analysis.png
# =========================================================================
logger.info("Đang vẽ Hình 36: 36_week7_hyperparameter_tuning_and_4class_analysis.png")
fig, axes = plt.subplots(1, 2, figsize=(16.5, 6.4))

# Panel A: Từng Fold (1..5) Recall trước và sau Tuning + In-Fold tau*_k cho Best Tree Model
sub_t_best = tuning_df[tuning_df["model"] == best_tree_name].sort_values("fold")
f_nums = sub_t_best["fold"].values
w = 0.34
b1 = axes[0].bar(f_nums - w / 2, sub_t_best["default_recall"], width=w, color="#bdc3c7", edgecolor="black", label="Recall Mặc định (tau=0.50)")
b2 = axes[0].bar(f_nums + w / 2, sub_t_best["tuned_recall"], width=w, color="#c0392b", edgecolor="black", label="Recall Sau Nested Tuning + In-Fold tau*_k")
axes[0].axhline(0.75, color="#27ae60", linestyle="--", linewidth=1.6, label="Mục tiêu Đề cương Recall >= 0.75")
for xp, d_r, t_r, tau_v in zip(f_nums, sub_t_best["default_recall"], sub_t_best["tuned_recall"], sub_t_best["tau_k"]):
    axes[0].text(xp - w / 2, d_r + 0.008, f"{d_r:.3f}", ha="center", fontsize=8.0, color="#555555", fontweight="bold")
    axes[0].text(xp + w / 2, t_r + 0.008, f"{t_r:.3f}\n(τ={tau_v:.2f})", ha="center", fontsize=7.8, color="#922b21", fontweight="bold")
axes[0].set_xticks(f_nums)
axes[0].set_xticklabels([f"Fold {f}" for f in f_nums])
axes[0].set_ylim(0.50, 0.90)
axes[0].set_xlabel("Lần kiểm thử (Test Fold k = 1..5)", fontsize=10.5, fontweight="bold")
axes[0].set_ylabel("Test Recall tại Mốc 25% Dynamic Active", fontsize=10.5, fontweight="bold")
axes[0].set_title(f"(A) Tác động Nested Tuning & In-Fold tau*_k qua 5 Fold ({best_tree_name})", fontsize=11.3, fontweight="bold")
axes[0].legend(loc="lower right", fontsize=8.5)
axes[0].grid(axis="y", alpha=0.3)

# Panel B: 4-Class OOF Confusion Matrix (100% sinh viên tại Mốc 25% Dynamic Active)
class_labels_4 = ["Distinction", "Pass", "Fail", "Withdrawn"]
cm4_25 = np.array(multiclass_cm_store["25%_LightGBM (4-Class)"])
cm4_pct = cm4_25 / cm4_25.sum(axis=1, keepdims=True) * 100.0
annot4 = np.array([[f"{cm4_25[r, c]:,}\n({cm4_pct[r, c]:.1f}%)" for c in range(4)] for r in range(4)])
sns.heatmap(
    cm4_pct, annot=annot4, fmt="", cmap="YlGnBu", cbar=False,
    xticklabels=class_labels_4, yticklabels=class_labels_4,
    linewidths=0.8, linecolor="black", annot_kws={"size": 9.2, "weight": "bold"}, ax=axes[1],
)
m4_row = multiclass_summary_df[(multiclass_summary_df["snapshot"] == "25%") & (multiclass_summary_df["model"] == "LightGBM (4-Class)")].iloc[0]
axes[1].set_title(
    f"(B) Ma trận Nhầm lẫn 4-Class OOF (25% Dynamic Active)\nAcc={m4_row['accuracy_4class_mean']:.3f} | Macro-F1={m4_row['macro_f1_4class_mean']:.3f} | Collapsed Binary AUC={m4_row['collapsed_binary_roc_auc_mean']:.3f}",
    fontsize=11.0, fontweight="bold",
)
axes[1].set_xlabel("Nhãn Dự đoán (4-Class)", fontsize=10.5, fontweight="bold")
axes[1].set_ylabel("Nhãn Thực tế (final_result)", fontsize=10.5, fontweight="bold")

plt.tight_layout()
fig.savefig(FIGURES_DIR / "36_week7_hyperparameter_tuning_and_4class_analysis.png", dpi=300)
plt.close(fig)

# =========================================================================
# HÌNH 37: 37_week7_10models_4milestones_benchmark.png
# =========================================================================
logger.info("Đang vẽ Hình 37: 37_week7_10models_4milestones_benchmark.png")
fig, axes = plt.subplots(1, 2, figsize=(18.8, 7.2))
x_snaps_10 = [25, 50, 75, 100]
c10_4 = combined_10_4snaps_df.copy()
c10_4["pct_num"] = c10_4["snapshot"].str.replace("%", "", regex=False).astype(int)

# Sắp xếp 10 mô hình theo AUC trung bình 4 mốc giảm dần
order_10 = (
    c10_4.groupby(["model", "family"], as_index=False)["roc_auc_mean"]
    .mean()
    .sort_values("roc_auc_mean", ascending=False)
    .reset_index(drop=True)
)

style_10_map = {
    "Ensemble / Stacking": ("#8e44ad", "D", "-", 2.5),
    "CatBoost / HistGBM": ("#e67e22", "^", "-", 2.3),
    "XGBoost": ("#27ae60", "s", "-", 2.3),
    "LightGBM": ("#2980b9", "o", "-", 2.3),
    "Neural Network (MLP)": ("#16a085", "P", "-", 2.0),
    "Logistic Regression": ("#2c3e50", "v", "--", 1.9),
    "Random Forest": ("#5d6d7e", "<", "--", 1.9),
    "SVM": ("#c0392b", ">", "-.", 1.7),
    "Decision Tree": ("#d35400", "X", ":", 1.8),
    "KNN": ("#7f8c8d", "*", ":", 1.9),
}

for _, row_o in order_10.iterrows():
    m_name = row_o["model"]
    fam_tag = "7B" if row_o["family"].startswith("Advanced") else "7A"
    col, mk, ls, lw = style_10_map[m_name]
    sub_m = c10_4[c10_4["model"] == m_name].sort_values("pct_num")
    axes[0].plot(
        x_snaps_10, sub_m["roc_auc_mean"],
        color=col, marker=mk, linestyle=ls, linewidth=lw, markersize=6.5,
        label=f"[{fam_tag}] {m_name} ({sub_m['roc_auc_mean'].iloc[0]:.3f} → {sub_m['roc_auc_mean'].iloc[-1]:.3f})",
    )

axes[0].axhline(0.80, color="#922b21", linestyle="--", linewidth=1.4, alpha=0.85, label="Mục tiêu Đề cương AUC = 0.80")
axes[0].set_xticks(x_snaps_10)
axes[0].set_xticklabels(["25%", "50%", "75%", "100%"])
axes[0].set_ylim(0.78, 0.985)
axes[0].set_xlabel("Mốc cắt Thời lượng Khóa học (%)", fontsize=10.5, fontweight="bold")
axes[0].set_ylabel("5-Fold Test ROC-AUC (Mean, tau = 0.50)", fontsize=10.5, fontweight="bold")
axes[0].set_title("(A) Tiến hóa ROC-AUC của 10 Mô hình qua 4 Mốc (Dynamic Active)", fontsize=11.6, fontweight="bold")
leg = axes[0].legend(bbox_to_anchor=(0.985, 0.13), loc="lower right", fontsize=7.9, framealpha=0.98, facecolor="white", ncol=1)
leg.set_zorder(10)
axes[0].grid(alpha=0.3)

# Panel B: Heatmap F1-Score của 10 Mô hình x 4 Mốc
heat_f1 = c10_4.pivot(index="model", columns="snapshot", values="f1_mean")[["25%", "50%", "75%", "100%"]]
heat_f1 = heat_f1.loc[order_10["model"].tolist()]
y_labels_heat = [
    f"[{'7B' if fam.startswith('Advanced') else '7A'}] {m}"
    for m, fam in zip(order_10["model"], order_10["family"])
]
sns.heatmap(
    heat_f1.values, annot=True, fmt=".4f", cmap="YlGnBu",
    xticklabels=["25%", "50%", "75%", "100%"], yticklabels=y_labels_heat,
    linewidths=0.8, linecolor="white", annot_kws={"size": 9.2, "weight": "bold"},
    cbar_kws={"label": "5-Fold F1-Score (Mean, tau = 0.50)"}, ax=axes[1],
)
axes[1].set_xlabel("Mốc cắt Thời lượng Khóa học (%)", fontsize=10.5, fontweight="bold")
axes[1].set_ylabel("Mô hình (Sắp theo ROC-AUC TB 4 mốc)", fontsize=10.5, fontweight="bold")
axes[1].set_title("(B) Heatmap 5-Fold F1-Score (10 Mô hình × 4 Mốc, Dynamic Active)", fontsize=11.6, fontweight="bold")

plt.tight_layout()
fig.savefig(FIGURES_DIR / "37_week7_10models_4milestones_benchmark.png", dpi=300)
plt.close(fig)

logger.info(f"Đã lưu mô hình và kết quả thực nghiệm Tuần 7 tại {MODELS_DIR} và {WEEK7_PROCESSED_DIR}!")

,model,tau_mean,recall_mean,recall_sd,f1_mean,f1_sd,roc_auc_mean,roc_auc_sd,pr_auc_mean
3,Ensemble / Stacking (Tuned),0.4140,0.7798,0.0212,0.7238,0.0096,0.8490,0.0035,0.8292
2,CatBoost / HistGBM (Tuned),0.4240,0.7792,0.0191,0.7253,0.0074,0.8482,0.0041,0.8275
1,XGBoost (Tuned),0.4320,0.7696,0.0111,0.7249,0.0067,0.8480,0.0031,0.8286
0,LightGBM (Tuned),0.4280,0.7712,0.0122,0.7243,0.0061,0.8477,0.0032,0.8281


2026-11-24 17:53:06,706 [INFO] Best Overall 5-Fold Model (25% Dynamic): Ensemble / Stacking (Tuned) (AUC=0.8490, F1=0.7238, Rec=0.7798)


2026-11-24 17:53:06,709 [INFO] Best Interpretable Tree Model (models/best_model.pkl): XGBoost (Tuned) (AUC=0.8480, F1=0.7249, Rec=0.7696, tau*=0.432)


2026-11-24 17:53:53,014 [INFO] Đang vẽ Hình 34: 34_week7_baseline_vs_advanced_5fold_benchmark.png


2026-11-24 17:53:53,764 [INFO] Đang vẽ Hình 35: 35_week7_early_prediction_5fold_progression_25_50_75_100.png


2026-11-24 17:53:54,377 [INFO] Đang vẽ Hình 36: 36_week7_hyperparameter_tuning_and_4class_analysis.png


2026-11-24 17:53:54,986 [INFO] Đang vẽ Hình 37: 37_week7_10models_4milestones_benchmark.png


2026-11-24 17:53:55,728 [INFO] Đã lưu mô hình và kết quả thực nghiệm Tuần 7 tại C:\Users\mrore\Project_I\models và C:\Users\mrore\Project_I\data\processed\week7_modeling!


## 7. Bộ Kiểm Thử Tự Động `TestWeek7AdvancedModels` (11/11 Unit Tests)
Kiểm định toàn bộ tiêu chí Definition of Done (DoD) của Tuần 7 trên kết quả 5-Fold Stratified Group CV:
- Không trùng sinh viên qua 5 Fold, không rò rỉ dữ liệu & ngưỡng In-Fold.
- Đủ 5 mô hình nâng cao + 5 mô hình Baseline (10 mô hình đối sánh) qua cả 4 mốc `25%–50%–75%–100%` + kiểm định thống kê.
- Tại mốc sớm **25% `dynamic_active`**, các mô hình Tuned đạt đồng thời **Mean Recall $\ge 0.75$**, **Mean F1 $\ge 0.70$**, **Mean AUC-ROC $\ge 0.80$**.
- Nạp lại `models/best_model.pkl` & `models/best_model_5fold_ensemble.joblib` thành công, kiểm tra đủ 4 biểu đồ `34`–`37` và các tệp kết quả tại `data/processed/week7_modeling/`.

In [5]:
class TestWeek7AdvancedModels(unittest.TestCase):
    @classmethod
    def setUpClass(cls):
        with open(WEEK7_PROCESSED_DIR / "week7_experiment_metrics.json", "r", encoding="utf-8") as f:
            cls.payload = json.load(f)
        cls.cv_df = pd.read_csv(WEEK7_PROCESSED_DIR / "week7_cv_results_summary.csv")
        cls.df_25_dyn = engine.filter_cohort(pd.read_csv(SNAPSHOTS_DIR / "features_snapshot_25pct.csv"), "dynamic_active")

    def test_01_all_5_advanced_models_evaluated_across_5_folds(self):
        pf = pd.DataFrame(self.payload["advanced_per_fold_results"])
        sub = pf[(pf["snapshot"] == "25%") & (pf["cohort"] == "dynamic_active")]
        expected = {"XGBoost", "LightGBM", "CatBoost / HistGBM", "Neural Network (MLP)", "Ensemble / Stacking"}
        self.assertEqual(set(sub["model"].unique()), expected)
        for m in expected:
            self.assertEqual(sorted(sub[sub["model"] == m]["fold"].tolist()), [1, 2, 3, 4, 5])

    def test_02_combined_10_models_and_statistical_tests_present(self):
        comb = pd.DataFrame(self.payload["combined_10_models_25pct_dynamic"])
        self.assertEqual(len(comb), 10)
        stats_df = pd.DataFrame(self.payload["statistical_tests_25pct_dynamic"])
        self.assertGreaterEqual(len(stats_df), 6)
        self.assertTrue((stats_df["delta_auc_mean"] > 0).all())

    def test_03_nested_tuning_and_infold_threshold_zero_leakage(self):
        t_df = pd.DataFrame(self.payload["nested_tuning_per_fold"])
        self.assertTrue((t_df["tau_k"] >= 0.15).all() and (t_df["tau_k"] <= 0.70).all())
        self.assertEqual(len(t_df), 15)  # 3 models x 5 folds

    def test_04_early_prediction_4_snapshots_complete(self):
        ew = pd.DataFrame(self.payload["early_prediction_summary"])
        dyn_ew = ew[ew["cohort"] == "dynamic_active"]
        self.assertEqual(set(dyn_ew["snapshot"].unique()), {"25%", "50%", "75%", "100%"})

    def test_05_dod_targets_met_at_25pct_dynamic_active_5fold(self):
        ew = pd.DataFrame(self.payload["early_prediction_summary"])
        row_25 = ew[
            (ew["snapshot"] == "25%")
            & (ew["cohort"] == "dynamic_active")
            & (ew["model"] == "LightGBM (Tuned)")
        ].iloc[0]
        self.assertGreaterEqual(float(row_25["recall_mean"]), 0.75, f"Recall 5-Fold = {row_25['recall_mean']} < 0.75")
        self.assertGreaterEqual(float(row_25["f1_mean"]), 0.70, f"F1 5-Fold = {row_25['f1_mean']} < 0.70")
        self.assertGreaterEqual(float(row_25["roc_auc_mean"]), 0.80, f"AUC 5-Fold = {row_25['roc_auc_mean']} < 0.80")

    def test_06_4class_experiment_and_collapsed_bridge_valid(self):
        mc = pd.DataFrame(self.payload["multiclass_4class_summary"])
        self.assertEqual(set(mc["snapshot"].unique()), {"25%", "100%"})
        self.assertTrue((mc["collapsed_binary_roc_auc_mean"] >= 0.82).all())

    def test_07_best_model_pkl_reloadable_and_predicts_114_features(self):
        model_pkl = joblib.load(MODELS_DIR / "best_model.pkl")
        bundle = joblib.load(MODELS_DIR / "best_model_5fold_ensemble.joblib")
        self.assertEqual(len(bundle["feature_names_114"]), 114)
        self.assertEqual(len(bundle["fold_models"]), 5)
        X_sample = engine.transform_with_state(self.df_25_dyn.head(10), bundle["full_preprocessor_state"])
        probs = model_pkl.predict_proba(X_sample)[:, 1]
        self.assertEqual(probs.shape, (10,))
        self.assertTrue((probs >= 0.0).all() and (probs <= 1.0).all())
        self.assertTrue(hasattr(model_pkl, "feature_importances_"))

    def test_08_metadata_and_cv_csv_valid(self):
        with open(MODELS_DIR / "best_model_metadata.json", "r", encoding="utf-8") as f:
            meta = json.load(f)
        self.assertEqual(meta["n_features_encoded"], 114)
        self.assertGreater(len(self.cv_df), 40)
        self.assertFalse(self.cv_df[["recall_mean", "f1_mean", "roc_auc_mean"]].isna().any().any())

    def test_09_no_forbidden_metrics_in_reports_dir(self):
        for fname in ["week7_experiment_metrics.json", "week7_baseline_cv_metrics.json", "week7_cv_results_summary.csv"]:
            self.assertFalse((REPORTS_DIR / fname).exists(), f"Vi phạm AGENTS.md: {fname} nằm trong reports/")
            self.assertTrue((WEEK7_PROCESSED_DIR / fname).exists())
        self.assertFalse(any(p.is_file() for p in PROCESSED_DIR.iterdir()), "Còn tệp rời tại gốc data/processed/!")

    def test_10_week7_figures_generated(self):
        for fig_name in [
            "34_week7_baseline_vs_advanced_5fold_benchmark.png",
            "35_week7_early_prediction_5fold_progression_25_50_75_100.png",
            "36_week7_hyperparameter_tuning_and_4class_analysis.png",
        ]:
            p = FIGURES_DIR / fig_name
            self.assertTrue(p.exists(), f"Thiếu biểu đồ: {fig_name}")
            self.assertGreater(p.stat().st_size, 10000)


    def test_11_all_10_models_4snapshots_benchmark_and_stats_complete(self):
        pf = pd.DataFrame(self.payload["advanced_per_fold_results"])
        pf_dyn = pf[pf["cohort"] == "dynamic_active"]
        self.assertEqual(set(pf_dyn["snapshot"].unique()), {"25%", "50%", "75%", "100%"})
        self.assertEqual(len(pf_dyn), 100)  # 4 snapshots x 5 models x 5 folds

        c4 = pd.DataFrame(self.payload["combined_10_models_4snapshots_dynamic"])
        self.assertEqual(len(c4), 40)  # 4 snapshots x 10 models
        self.assertEqual(set(c4["snapshot"].unique()), {"25%", "50%", "75%", "100%"})
        self.assertFalse(c4[["accuracy_mean", "precision_mean", "recall_mean", "f1_mean", "roc_auc_mean", "pr_auc_mean"]].isna().any().any())

        st4 = pd.DataFrame(self.payload["statistical_tests_4snapshots_dynamic"])
        self.assertEqual(len(st4), 32)  # 4 snapshots x 8 pairs
        self.assertEqual(set(st4["snapshot"].unique()), {"25%", "50%", "75%", "100%"})
        self.assertTrue((st4["delta_auc_mean"] > 0).all())

        fig37 = FIGURES_DIR / "37_week7_10models_4milestones_benchmark.png"
        self.assertTrue(fig37.exists(), "Thiếu biểu đồ 37_week7_10models_4milestones_benchmark.png")
        self.assertGreater(fig37.stat().st_size, 10000)


suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestWeek7AdvancedModels)
result = unittest.TextTestRunner(stream=sys.stdout, verbosity=2).run(suite)
assert result.wasSuccessful(), f"Có test thất bại: failures={len(result.failures)}, errors={len(result.errors)}"
print(f"\n[PASSED] {result.testsRun}/{result.testsRun} kiểm thử Giai đoạn 7B (06_advanced_models.ipynb) hoàn tất không lỗi!")

test_01_all_5_advanced_models_evaluated_across_5_folds (__main__.TestWeek7AdvancedModels.test_01_all_5_advanced_models_evaluated_across_5_folds) ... 

ok


test_02_combined_10_models_and_statistical_tests_present (__main__.TestWeek7AdvancedModels.test_02_combined_10_models_and_statistical_tests_present) ... 

ok


test_03_nested_tuning_and_infold_threshold_zero_leakage (__main__.TestWeek7AdvancedModels.test_03_nested_tuning_and_infold_threshold_zero_leakage) ... 

ok


... (14 outputs left out)

test_11_all_10_models_4snapshots_benchmark_and_stats_complete (__main__.TestWeek7AdvancedModels.test_11_all_10_models_4snapshots_benchmark_and_stats_complete) ... 

ok


----------------------------------------------------------------------
Ran 11 tests in 0.661s

OK



[PASSED] 11/11 kiểm thử Giai đoạn 7B (06_advanced_models.ipynb) hoàn tất không lỗi!
